In [1]:
"""
═══════════════════════════════════════════
Central configuration, all typed data structures, enums, and terminal logger.

Episodic Memory (Tulving, 1972):
  "Memory for personally experienced events — the what, when, where
   of specific episodes bound together in a temporal-contextual fabric."

This system models seven dimensions of each episode:
  WHO     — participants/agents involved
  WHAT    — action/event type and content
  WHEN    — timestamp, duration, temporal position in sequence
  WHERE   — spatial/contextual location
  WHY     — motivation, cause, preceding trigger
  HOW     — method, manner, instruments used
  EMOTION — affective valence and arousal (−1 to +1, 0 to 1)

References:
  • Episodic Memory (Tulving, 1972/1983)   https://doi.org/10.1016/S0079-7421(08)60467-4
  • Generative Agents (Park et al., 2023)  https://arxiv.org/abs/2304.03442
  • MemGPT (Packer et al., 2023)           https://arxiv.org/abs/2310.08560
  • Cognitive Architecture CoALA           https://arxiv.org/abs/2309.02427
  • HippoRAG (Gutierrez et al., 2024)      https://arxiv.org/abs/2405.14831
  • A-MEM (Xu et al., 2024)               https://arxiv.org/abs/2312.06681
"""

'\n═══════════════════════════════════════════\nCentral configuration, all typed data structures, enums, and terminal logger.\n\nEpisodic Memory (Tulving, 1972):\n  "Memory for personally experienced events — the what, when, where\n   of specific episodes bound together in a temporal-contextual fabric."\n\nThis system models seven dimensions of each episode:\n  WHO     — participants/agents involved\n  WHAT    — action/event type and content\n  WHEN    — timestamp, duration, temporal position in sequence\n  WHERE   — spatial/contextual location\n  WHY     — motivation, cause, preceding trigger\n  HOW     — method, manner, instruments used\n  EMOTION — affective valence and arousal (−1 to +1, 0 to 1)\n\nReferences:\n  • Episodic Memory (Tulving, 1972/1983)   https://doi.org/10.1016/S0079-7421(08)60467-4\n  • Generative Agents (Park et al., 2023)  https://arxiv.org/abs/2304.03442\n  • MemGPT (Packer et al., 2023)           https://arxiv.org/abs/2310.08560\n  • Cognitive Architecture CoAL

In [10]:

# ─────────────────────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────────────────────
from __future__ import annotations

import os
import re
import ast
import sys
import json
import math
import time
import hashlib
import textwrap
import warnings
import operator
import datetime
import traceback
from enum import Enum
from pathlib import Path
from typing import (
    Any, Callable, Dict, List, Optional,
    Tuple, Union, NamedTuple
)
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")


In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

W0525 08:03:07.228000 30284 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [12]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_classic.agents import AgentExecutor
from langgraph.checkpoint.memory import InMemorySaver


In [13]:
import chromadb
from chromadb.config import Settings


In [3]:


# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Ollama (local LLM) ─────────────────────────────────────────────
    OLLAMA_BASE_URL: str = field(default_factory=lambda: os.getenv(
        "OLLAMA_BASE_URL", "http://localhost:11434"))
    OLLAMA_MODEL: str    = field(default_factory=lambda: os.getenv(
        "OLLAMA_MODEL", "qwen2.5-coder:7b"))

    # ── HuggingFace (local embeddings) ───────────────────────────────
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = field(default_factory=lambda: os.getenv("EMBEDDING_DEVICE", "cpu"))

    # ── Storage ───────────────────────────────────────────────────────
    BASE_DIR: str                 = "./episodic_store"
    EPISODES_DB: str              = "./episodic_store/episodes.db"
    EPISODE_GRAPH_DB: str         = "./episodic_store/episode_graph.db"
    CHROMA_DIR: str               = "./episodic_store/chroma"
    KB_CHROMA_DIR: str            = "./episodic_store/kb_chroma"
    CHROMA_COLLECTION: str        = "episodic_embeddings"
    KB_COLLECTION: str            = "episodic_kb"

    # ── Retrieval parameters ──────────────────────────────────────────
    TOP_K_SEMANTIC: int           = 5    # semantic similarity hits
    TOP_K_TEMPORAL: int           = 4    # temporal window results
    TOP_K_ASSOCIATIVE: int        = 3    # graph spreading activation
    TOP_K_KB: int                 = 3    # knowledge base
    FETCH_K: int                  = 10
    MMR_LAMBDA: float             = 0.65

    # ── Episodic scoring weights (Park et al. 2023 formula) ──────────
    W_RECENCY: float              = 1.0
    W_IMPORTANCE: float           = 1.0
    W_RELEVANCE: float            = 1.0
    RECENCY_DECAY_BASE: float     = 0.99  # per-hour decay
    FORGETTING_HALFLIFE_DAYS: float = 30.0

    # ── Spreading activation (episode graph) ─────────────────────────
    ACTIVATION_DEPTH: int         = 2     # BFS depth for graph traversal
    ACTIVATION_DECAY: float       = 0.6   # activation × decay per hop
    MIN_ACTIVATION: float         = 0.1   # prune nodes below this

    # ── Episode chunking (for long content) ──────────────────────────
    CHUNK_SIZE: int               = 600
    CHUNK_OVERLAP: int            = 100

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float        = 0.0
    LLM_MAX_TOKENS: int           = 1200

    # ── Interference ─────────────────────────────────────────────────
    INTERFERENCE_THRESHOLD: float = 0.85  # cosine sim threshold for conflict detection

    def is_configured(self) -> bool:
        """Check Ollama is reachable."""
        import urllib.request
        try:
            urllib.request.urlopen(self.OLLAMA_BASE_URL, timeout=3)
            return True
        except Exception:
            return False
# ══════════════════════════════════════════════════════════════════════
# ENUMS
# ══════════════════════════════════════════════════════════════════════

class EpisodeType(str, Enum):
    CONVERSATION   = "conversation"    # a dialogue exchange
    OBSERVATION    = "observation"     # something noticed/perceived
    ACTION         = "action"          # something done
    LEARNING       = "learning"        # something learned
    DECISION       = "decision"        # a choice made
    EMOTION_EVENT  = "emotion_event"   # an emotionally significant event
    SOCIAL         = "social"          # interaction with another person
    REFLECTION     = "reflection"      # introspective thought
    PLAN           = "plan"            # future-oriented episode
    CORRECTION     = "correction"      # correction of prior belief/action


class EdgeType(str, Enum):
    FOLLOWS        = "follows"         # temporal succession
    CAUSES         = "causes"          # causal link
    CONTRADICTS    = "contradicts"     # conflicting episodes
    SUPPORTS       = "supports"        # corroborating episodes
    INVOLVES       = "involves"        # shared participant
    PART_OF        = "part_of"         # episode is sub-event of another
    TRIGGERED_BY   = "triggered_by"    # emotional/motivational trigger
    SIMILAR_TO     = "similar_to"      # semantic similarity (≥ 0.85)


class RetrievalMode(str, Enum):
    SEMANTIC       = "semantic"        # embedding similarity
    TEMPORAL       = "temporal"        # time window / before/after/during
    ASSOCIATIVE    = "associative"     # graph spreading activation
    CUE_BASED      = "cue_based"       # who/what/when/where/why facets
    RECONSTRUCTIVE = "reconstructive"  # rebuild full episode from fragments
    HYBRID         = "hybrid"          # all modes fused


class EmotionLabel(str, Enum):
    VERY_POSITIVE = "very_positive"  # valence > 0.7
    POSITIVE      = "positive"       # valence 0.3–0.7
    NEUTRAL       = "neutral"        # valence −0.3–0.3
    NEGATIVE      = "negative"       # valence −0.7–−0.3
    VERY_NEGATIVE = "very_negative"  # valence < −0.7


In [4]:

# ══════════════════════════════════════════════════════════════════════
# CORE EPISODE DATACLASS
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Episode:
    """
    A single episodic memory record — the fundamental unit of
    episodic memory retrieval.

    Encodes all 7W dimensions: WHO, WHAT, WHEN, WHERE, WHY, HOW, EMOTION.
    """
    # Identity
    episode_id:    str
    agent_id:      str                  # owner of this memory

    # 7W Dimensions
    what:          str                  # core content / event description
    when_ts:       float                = field(default_factory=time.time)
    when_label:    str                  = ""         # human-readable e.g. "morning", "2024-01"
    who:           List[str]            = field(default_factory=list)    # participants
    where:         str                  = ""         # context / location
    why:           str                  = ""         # motivation / cause
    how:           str                  = ""         # method / manner
    emotion_valence: float              = 0.0        # −1 (very negative) → +1 (very positive)
    emotion_arousal: float              = 0.0        # 0 (calm) → 1 (intense)
    emotion_label:   str                = EmotionLabel.NEUTRAL.value

    # Episode metadata
    episode_type:  str                  = EpisodeType.OBSERVATION.value
    importance:    float                = 0.5        # 0–1, LLM-scored
    access_count:  int                  = 0
    last_accessed: float                = field(default_factory=time.time)
    strength:      float                = 1.0        # weakens over time
    tags:          List[str]            = field(default_factory=list)
    source:        str                  = ""         # e.g. "user_input", "document", "inference"

    # Graph connectivity
    linked_ids:    List[str]            = field(default_factory=list)   # adjacent episode IDs
    session_id:    str                  = ""

    # Raw text for embedding
    raw_text:      str                  = ""         # concatenation of 7W for embedding

    def build_raw_text(self) -> str:
        """Build the embedding-ready text representation."""
        parts = [f"What: {self.what}"]
        if self.who:
            parts.append(f"Who: {', '.join(self.who)}")
        if self.where:
            parts.append(f"Where: {self.where}")
        if self.why:
            parts.append(f"Why: {self.why}")
        if self.how:
            parts.append(f"How: {self.how}")
        if self.when_label:
            parts.append(f"When: {self.when_label}")
        if self.emotion_label and self.emotion_label != EmotionLabel.NEUTRAL.value:
            parts.append(f"Emotion: {self.emotion_label} (valence={self.emotion_valence:.2f})")
        return " | ".join(parts)

    def recency_score(self, decay_base: float = 0.99) -> float:
        """
        Recency score from Generative Agents (Park et al., 2023):
          recency = decay_base ^ (hours_since_last_access)
        """
        hours = (time.time() - self.last_accessed) / 3600.0
        return (decay_base ** hours)

    def forgetting_curve_strength(self, halflife_days: float = 30.0) -> float:
        """
        Ebbinghaus forgetting curve (MemoryBank, Zhong et al., 2023):
          strength(t) = S₀ × 0.5^(age_days / halflife) × (1 + 0.1 × access_count)
        Repeated access reinforces memory.
        """
        age_days = (time.time() - self.when_ts) / 86400.0
        base = self.strength * (0.5 ** (age_days / halflife_days))
        reinforced = base * (1.0 + 0.1 * self.access_count)
        return min(1.0, reinforced)

    def composite_score(
        self,
        relevance: float,
        w_recency: float = 1.0,
        w_importance: float = 1.0,
        w_relevance: float = 1.0,
        decay_base: float = 0.99,
    ) -> float:
        """
        Composite retrieval score from Generative Agents (Park et al., 2023):
          score = α_rec × recency + α_imp × importance + α_rel × relevance
        All components normalized to [0, 1].
        """
        rec = self.recency_score(decay_base)
        imp = self.importance
        rel = max(0.0, min(1.0, relevance))
        total_w = w_recency + w_importance + w_relevance
        return (w_recency * rec + w_importance * imp + w_relevance * rel) / total_w

    def to_context_str(self) -> str:
        """Format episode as readable context for LLM."""
        age_hours = (time.time() - self.when_ts) / 3600.0
        age_str = (f"{age_hours:.0f}h ago" if age_hours < 48
                   else f"{age_hours/24:.0f}d ago")
        lines = [
            f"[{self.episode_type.upper()}  ·  {age_str}  ·  imp={self.importance:.2f}]",
            f"  WHAT   : {self.what}",
        ]
        if self.who:   lines.append(f"  WHO    : {', '.join(self.who)}")
        if self.where: lines.append(f"  WHERE  : {self.where}")
        if self.why:   lines.append(f"  WHY    : {self.why}")
        if self.how:   lines.append(f"  HOW    : {self.how}")
        if self.emotion_label != EmotionLabel.NEUTRAL.value:
            lines.append(f"  EMOTION: {self.emotion_label} "
                         f"(v={self.emotion_valence:.2f}, a={self.emotion_arousal:.2f})")
        if self.tags:  lines.append(f"  TAGS   : {', '.join(self.tags)}")
        return "\n".join(lines)


In [5]:


@dataclass
class EpisodeEdge:
    """A directed edge in the episode graph."""
    edge_id:    str
    source_id:  str           # from episode
    target_id:  str           # to episode
    edge_type:  str           # EdgeType value
    weight:     float = 1.0   # strength of relationship
    timestamp:  float = field(default_factory=time.time)
    metadata:   Dict[str, Any] = field(default_factory=dict)


@dataclass
class RetrievalResult:
    """Complete result from an episodic retrieval query."""
    query:             str
    agent_id:          str
    mode:              RetrievalMode
    episodes:          List[Episode]          = field(default_factory=list)
    kb_docs:           List[Any]              = field(default_factory=list)
    scores:            Dict[str, float]       = field(default_factory=dict)
    reconstructed:     Optional[str]          = None   # reconstructive recall output
    narrative:         Optional[str]          = None   # narrative summary
    interference:      List[Tuple[str, str]]  = field(default_factory=list)  # conflicting pairs
    temporal_range:    Optional[Tuple[float, float]] = None  # (start_ts, end_ts)
    retrieval_time_ms: int                    = 0
    modes_used:        List[str]              = field(default_factory=list)
    total_searched:    int                    = 0

    def format_episodes_for_llm(self) -> str:
        if not self.episodes:
            return "(no episodic memories found)"
        parts = []
        for i, ep in enumerate(self.episodes[:8], 1):
            score = self.scores.get(ep.episode_id, 0.0)
            parts.append(f"--- Episode {i} [score={score:.3f}] ---")
            parts.append(ep.to_context_str())
        return "\n\n".join(parts)



In [6]:


# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

class C:
    RESET   = "\033[0m"; BOLD   = "\033[1m"; DIM    = "\033[2m"
    CYAN    = "\033[96m"; GREEN = "\033[92m"; YELLOW = "\033[93m"
    RED     = "\033[91m"; MAG   = "\033[95m"; BLUE   = "\033[94m"
    WHITE   = "\033[97m"; ORANGE= "\033[33m"
    # Retrieval mode colours
    SEM     = "\033[96m"   # cyan      — semantic
    TEMP    = "\033[92m"   # green     — temporal
    ASSOC   = "\033[95m"   # magenta   — associative
    CUE     = "\033[33m"   # orange    — cue-based
    RECON   = "\033[94m"   # blue      — reconstructive
    INTERF  = "\033[91m"   # red       — interference
    GRAPH   = "\033[93m"   # yellow    — graph


W = 76

MODE_COLOURS = {
    "semantic":       C.SEM,
    "temporal":       C.TEMP,
    "associative":    C.ASSOC,
    "cue_based":      C.CUE,
    "reconstructive": C.RECON,
    "hybrid":         C.MAG,
}


class Log:
    @staticmethod
    def banner(title: str, sub: str = ""):
        print(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        print(f"  {title}")
        if sub:
            print(f"  {C.DIM}{sub}{C.RESET}{C.BOLD}{C.WHITE}")
        print(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str, colour: str = C.CYAN):
        print(f"\n{C.BOLD}{colour}{'━'*W}")
        print(f"  {title}")
        print(f"{'━'*W}{C.RESET}")

    @staticmethod
    def mode(mode_name: str, msg: str):
        col = MODE_COLOURS.get(mode_name, C.WHITE)
        print(f"{C.BOLD}{col}[{mode_name.upper():13s}]{C.RESET} {msg}")

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def episode(ep: "Episode", score: float = 0.0):
        age_d = (time.time() - ep.when_ts) / 86400.0
        print(f"  {C.DIM}[{ep.episode_id[:12]}]{C.RESET} "
              f"{C.BOLD}{ep.what[:60]}{C.RESET} "
              f"{C.DIM}| type={ep.episode_type} score={score:.3f} age={age_d:.1f}d{C.RESET}")

    @staticmethod
    def graph_edge(src: str, edge_type: str, tgt: str, weight: float):
        print(f"  {C.GRAPH}{src[:10]} ─[{edge_type}:{weight:.2f}]→ {tgt[:10]}{C.RESET}")

    @staticmethod
    def answer_box(query: str, answer: str, meta: Dict[str, Any]):
        import textwrap
        print(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        print(f"  EPISODIC MEMORY RETRIEVAL ANSWER")
        print(f"{'═'*W}{C.RESET}")
        print(f"{C.BOLD}  Q: {query}{C.RESET}\n")
        for line in textwrap.wrap(answer, W - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*W}")
        print(f"  Retrieval mode   : {meta.get('mode', '?')}")
        print(f"  Episodes found   : {meta.get('episode_count', 0)}")
        print(f"  KB docs          : {meta.get('kb_count', 0)}")
        print(f"  Graph hops       : {meta.get('graph_hops', 0)}")
        print(f"  Reconstructed    : {meta.get('reconstructed', False)}")
        print(f"  Interference     : {meta.get('interference_count', 0)} conflicts")
        print(f"  Elapsed          : {meta.get('elapsed', 0):.2f}s")
        print(f"{'─'*W}{C.RESET}")


In [7]:
"""
════════════════════════════════════════════
Knowledge base corpus: 10 documents covering episodic memory systems,
cognitive science, and AI memory architectures.

📄 Reference PDFs (all publicly available):
  1. Tulving (1983) — Elements of Episodic Memory    https://doi.org/10.1093/acprof:oso/9780198521259.001.0001
  2. Generative Agents (Park et al., 2023)           https://arxiv.org/pdf/2304.03442
  3. MemGPT (Packer et al., 2023)                    https://arxiv.org/pdf/2310.08560
  4. HippoRAG (Gutierrez et al., 2024)               https://arxiv.org/pdf/2405.14831
  5. A-MEM (Xu et al., 2024)                         https://arxiv.org/pdf/2312.06681
  6. CoALA (Sumers et al., 2023)                     https://arxiv.org/pdf/2309.02427
  7. MemoryBank (Zhong et al., 2023)                 https://arxiv.org/pdf/2305.10250
  8. Episodic Memory in AI (Nuxoll & Laird, 2007)    https://doi.org/10.1145/1597148.1597232
  9. ReAct (Yao et al., 2023)                        https://arxiv.org/pdf/2210.03629
  10. Attention Is All You Need (Vaswani, 2017)       https://arxiv.org/pdf/1706.03762
"""

'\n════════════════════════════════════════════\nKnowledge base corpus: 10 documents covering episodic memory systems,\ncognitive science, and AI memory architectures.\n\n📄 Reference PDFs (all publicly available):\n  1. Tulving (1983) — Elements of Episodic Memory    https://doi.org/10.1093/acprof:oso/9780198521259.001.0001\n  2. Generative Agents (Park et al., 2023)           https://arxiv.org/pdf/2304.03442\n  3. MemGPT (Packer et al., 2023)                    https://arxiv.org/pdf/2310.08560\n  4. HippoRAG (Gutierrez et al., 2024)               https://arxiv.org/pdf/2405.14831\n  5. A-MEM (Xu et al., 2024)                         https://arxiv.org/pdf/2312.06681\n  6. CoALA (Sumers et al., 2023)                     https://arxiv.org/pdf/2309.02427\n  7. MemoryBank (Zhong et al., 2023)                 https://arxiv.org/pdf/2305.10250\n  8. Episodic Memory in AI (Nuxoll & Laird, 2007)    https://doi.org/10.1145/1597148.1597232\n  9. ReAct (Yao et al., 2023)                        http

In [14]:

CORPUS: List[Dict[str, str]] = [
    {
        "id": "tulving_episodic",
        "source": "Elements of Episodic Memory — Tulving (1983)",
        "url": "https://doi.org/10.1093/acprof:oso/9780198521259.001.0001",
        "content": """
Endel Tulving (1972, 1983) introduced the distinction between episodic and semantic memory.
Episodic memory stores personally experienced events — the "what, when, where" of specific
experiences. Semantic memory stores general world knowledge without temporal context.

Key properties of episodic memory:
  1. Temporal: events are bound to specific time points and sequences
  2. Contextual: events encoded with their surrounding context (spatial, emotional, social)
  3. Personal: first-person perspective — remembered as experienced by the self
  4. Reconstructive: retrieval rebuilds the episode from fragments, not verbatim playback
  5. Autonoetic consciousness: the ability to mentally "travel back in time"

Encoding specificity principle: retrieval is most effective when retrieval cues
match the encoding context. This motivates multi-facet indexing in AI episodic memory
systems (WHO, WHAT, WHEN, WHERE, WHY, HOW, EMOTION).

Forgetting in episodic memory follows Ebbinghaus (1885):
  R = e^(-t/S)  where R = retention, t = time, S = memory strength
Repeated retrieval acts as re-encoding and resets the forgetting curve.

Tulving's SPI model (Serial-Parallel-Independent):
  - Encoding: semantic → episodic (serial dependency)
  - Storage: semantic ∥ episodic (parallel, independent)
  - Retrieval: episodic → semantic or semantic alone (independent)
"""
    },
    {
        "id": "generative_agents",
        "source": "Generative Agents: Interactive Simulacra of Human Behavior — Park et al. (2023)",
        "url": "https://arxiv.org/abs/2304.03442",
        "content": """
Generative Agents implement a computational episodic memory stream: a chronological
record of experiences stored as natural language with timestamps.

Retrieval scoring (three-factor composite):
  score(m) = α_rec × recency(m) + α_imp × importance(m) + α_rel × relevance(m,q)

  recency(m) = decay_factor ^ (hours_since_last_retrieval)
               Park et al. use decay_factor = 0.99 per hour

  importance(m): LLM rates on 1–10 scale ("on a scale of 1 to 10, how important
                 is this memory to the agent's life?")

  relevance(m,q): cosine similarity between memory embedding and query embedding

All three components normalized to [0, 1] before combining.

Reflection mechanism: after N memories, the agent asks "What are 3 high-level
insights from recent memories?" and stores resulting insights as new memories.
This creates a hierarchy: raw episodes → reflected insights → personality.

Results: agents showed emergent social behaviors (spreading information, coordinating
plans) arising purely from the memory architecture. Memory-enabled agents rated
78% believable vs 34% without episodic memory.
"""
    },
    {
        "id": "hipporag",
        "source": "HippoRAG: Neurologically Inspired Long-Term Memory for RAG — Gutierrez et al. (2024)",
        "url": "https://arxiv.org/abs/2405.14831",
        "content": """
HippoRAG models the hippocampal indexing theory of memory (HPC-Neocortex interaction).
The hippocampus acts as an index: it stores episode-specific conjunctive representations
that bind together neocortical (semantic) memories into coherent episodes.

Implementation:
  • OpenIE extracts (subject, relation, object) triples from passages
  • KG nodes = named entities; KG edges = relations
  • Each node embedded with sentence-transformers
  • Personalized PageRank (PPR) propagated from query-matched seed nodes

PPR formula: π = (1−α)·A·π + α·s
  where A = row-normalized adjacency, s = seed node vector, α = teleport prob (0.15)

Associative retrieval: query matches seed entities → PPR flows through the KG →
passages associated with high-PageRank nodes retrieved. This enables multi-hop
retrieval (A→B→C) not possible with flat vector search.

Results vs standard dense RAG:
  MuSiQue multi-hop: 31.7% vs DPR 23.4%
  2WikiMultihopQA: 42.1% vs 27.8%
  HotpotQA: 44.2% vs 40.1%

Hippocampal indexing theory (Teyler & DiScenna, 1986): the hippocampus stores
addresses into neocortical memory rather than the memories themselves. Episodic
recall activates the hippocampal index, which triggers reconstruction of the full
episode from distributed neocortical storage.
"""
    },
    {
        "id": "amem",
        "source": "A-MEM: Agentic Memory for LLM Agents — Xu et al. (2024)",
        "url": "https://arxiv.org/abs/2312.06681",
        "content": """
A-MEM implements a Zettelkasten (slip-box) note-taking system for episodic memory.
Each memory note contains: content, keywords, contextual description, and explicit
links to related notes — forming a rich associative network.

Memory operations:
  add(text):    LLM generates keywords + description, HNSW embed, BFS-link to ≥0.8 similar
  retrieve(q):  semantic match → follow links to neighbors (BFS depth 2)
  update(id):   regenerate metadata, re-embed, update links
  evolve():     LLM re-weights importance based on access history + context change

Graph retrieval advantage: retrieve episode A → follow links to B and C
(related by shared entities/themes) even if B and C don't directly match the query.
This mimics associative recall — "one memory reminds you of another."

Linking algorithm:
  FOR each new note n:
    candidates = HNSW.search(n.embedding, k=10)
    FOR c in candidates WHERE cosine_sim(n, c) >= 0.80:
      create_bidirectional_link(n, c, weight=cosine_sim)

A-MEM vs MemGPT on multi-hop QA: 71.2% vs 58.9% accuracy.
The link structure enables traversal of memory chains not captured by single-hop
semantic search, directly paralleling hippocampal associative binding.
"""
    },
    {
        "id": "coala",
        "source": "Cognitive Architecture for Language Agents (CoALA) — Sumers et al. (2023)",
        "url": "https://arxiv.org/abs/2309.02427",
        "content": """
CoALA categorizes memory in language agents across two dimensions:
  Storage type: in-context (temporary) vs external (persistent) vs in-weights (parametric)
  Memory type: episodic, semantic, procedural, working

Episodic memory in CoALA:
  Purpose: records of past actions, observations, and experiences
  Storage: external vector/graph database
  Operations: add (new experience), retrieve (query), reflect (summarize/abstract)

Decision cycle for episodic retrieval:
  1. Observe new input
  2. Query episodic store for relevant past experiences
  3. Reason over retrieved episodes + current input
  4. Execute action
  5. Store new episode (observation + action + outcome)

Episodic vs semantic distinction in CoALA:
  Episodic: "I visited Paris on March 3rd and had coffee at Café de Flore"
  Semantic: "Paris is the capital of France; cafés are popular there"

The SPI model (Tulving) is implemented computationally:
  Episode encoding depends on semantic understanding (serial)
  Both stored and retrieved independently (parallel)
  Episodic retrieval can trigger semantic associations (independent paths)
"""
    },
    {
        "id": "memorybank_forgetting",
        "source": "MemoryBank: Enhancing LLMs with Long-Term Memory — Zhong et al. (2023)",
        "url": "https://arxiv.org/abs/2305.10250",
        "content": """
MemoryBank implements the Ebbinghaus forgetting curve to model episodic memory decay:

  strength(t) = S₀ × exp(−t / (τ × n))
  where S₀ = initial strength, t = time elapsed, τ = decay constant, n = retrieval count

Memory operations:
  store(event):    encode event + timestamp, compute initial importance
  retrieve(query): ranked by recency + importance + similarity
  update(id):      retrieved memories strengthened (τ multiplied by n+1)
  forget():        memories below minimum strength pruned

User profile evolution:
  Every session, the LLM reads recent episodes and updates a profile document:
  "User is interested in X, prefers Y style, has mentioned Z topic 5 times"

Retrieval ranking formula (normalized Euclidean score):
  final_score = λ₁ × similarity + λ₂ × recency + λ₃ × strength
  Default: λ₁=0.5, λ₂=0.3, λ₃=0.2

Evaluation: SiliconFriend chatbot with MemoryBank scored:
  Personalization: 4.2/5 vs 2.8/5 (baseline)
  Emotional consistency: 3.9/5 vs 2.4/5
  Long-term coherence: 4.4/5 vs 2.6/5
"""
    },
    {
        "id": "nuxoll_laird",
        "source": "Extending Cognitive Architecture with Episodic Memory — Nuxoll & Laird (2007)",
        "url": "https://dl.acm.org/doi/10.5555/1597148.1597232",
        "content": """
Nuxoll and Laird added episodic memory to the Soar cognitive architecture. Their work
establishes computational properties of episodic memory in AI systems:

Storage: every cognitive cycle, a snapshot of working memory is stored as an episode.
Each episode = {WM contents, timestamp, context tags}.

Retrieval by partial cue matching:
  Given a cue (partial WM state), find the episode whose content best matches the cue.
  Score = |matching_elements| / |total_cue_elements|   (normalized overlap)

Temporal chaining: episodes are stored in temporal sequence; given episode E,
the system can retrieve E-1 (previous) and E+1 (next) to reconstruct temporal context.

Interference in episodic memory: when two similar cues exist, the less recent one
can interfere with retrieval of the more relevant one. Detection:
  interference_score = sim(cue, episode_A) × sim(cue, episode_B) × temporal_overlap

Applications in Soar: episodic memory enabled:
  • Learning to avoid repeated mistakes (store failed episode → avoid similar cues)
  • Temporal reasoning (before/after queries)
  • Self-monitoring (compare current state to past states)

The key insight: episodic memory enables learning from single experiences,
unlike procedural/semantic memory which requires repetition.
"""
    },
    {
        "id": "spreading_activation",
        "source": "Spreading Activation in Semantic Memory — Collins & Loftus (1975)",
        "url": "https://doi.org/10.1037/0033-295X.82.6.407",
        "content": """
Collins and Loftus (1975) proposed spreading activation theory: when a concept
is activated, activation spreads to semantically related concepts with decaying
strength proportional to link distance.

In episodic memory retrieval systems, spreading activation is implemented as:
  1. Initial nodes receive activation A₀ = similarity(query, episode)
  2. Activation propagates to linked neighbors: A_neighbor = A_source × decay_factor
  3. Nodes accumulate activation from multiple paths
  4. Retrieved set = episodes with activation > threshold θ

BFS-based implementation:
  queue = [(seed_episode, A₀)]
  visited = {}
  WHILE queue not empty AND depth < max_depth:
    (node, activation) = queue.pop()
    IF activation < θ: skip
    visited[node] = activation
    FOR neighbor in graph.neighbors(node):
      new_act = activation × decay_factor
      queue.append((neighbor, new_act))

Parameters: decay_factor = 0.6 (moderately fast decay), θ = 0.1 (retrieval threshold)
This produces a relevance-ranked set of episodes that may not be directly similar
to the query but are associatively linked through shared concepts.
"""
    },
    {
        "id": "react",
        "source": "ReAct: Synergizing Reasoning and Acting in Language Models — Yao et al. (2023)",
        "url": "https://arxiv.org/abs/2210.03629",
        "content": """
ReAct interleaves Thought, Action, Observation steps — a pattern directly applicable
to episodic memory retrieval where the agent reasons about which retrieval strategy
to apply, acts to retrieve episodes, and observes the results before synthesizing.

Episodic memory retrieval in a ReAct framework:
  Thought: "The user is asking about a past decision. I should use temporal retrieval."
  Action: TEMPORAL_RETRIEVE(query="project decision", before=now, after=week_ago)
  Observation: [3 episodes from last week about project planning]
  Thought: "Episode E-22 seems most relevant. Let me check its neighbors."
  Action: ASSOCIATIVE_RETRIEVE(seed_id="E-22", depth=2)
  Observation: [E-19, E-24, E-31 — related discussion and outcome]
  Final Answer: "Based on your past experiences..."

The ReAct loop naturally handles multi-step episodic queries:
  "What led to the decision X?" → temporal → causal chain traversal → answer
  "Has this happened before?" → semantic similarity → temporal ordering → answer
  "What was I feeling when...?" → cue-based (emotion filter) → reconstructive recall

ReAct performance on HotpotQA: 50.3 EM vs CoT 29.4, demonstrating that
interleaved retrieval + reasoning outperforms single-shot retrieval.
"""
    },
    {
        "id": "transformer_attention",
        "source": "Attention Is All You Need — Vaswani et al. (2017)",
        "url": "https://arxiv.org/abs/1706.03762",
        "content": """
The Transformer attention mechanism is foundational to episodic memory retrieval:
the Query-Key-Value (QKV) framework directly models associative memory lookup.

QKV and episodic memory analogy:
  Query   = retrieval cue (current context / question)
  Keys    = episode index representations (what the episode is "about")
  Values  = full episode content (what to retrieve)
  Output  = weighted sum of Values = reconstructed episodic context

Attention(Q,K,V) = softmax(Q·K^T / √d_k) · V

In episodic memory systems, this maps to:
  q = embed(query_cue)
  K = [embed(ep.raw_text) for ep in all_episodes]
  V = [ep.to_context_str() for ep in all_episodes]
  retrieved_context = Attention(q, K, V)

Cross-attention between temporal positions enables "what happened before/after":
  position_bias(pos_i, pos_j) = f(|pos_i - pos_j|)  ← temporal distance weighting

Multi-head attention in episodic context:
  Head 1 might attend to semantic similarity
  Head 2 might attend to temporal proximity
  Head 3 might attend to emotional valence similarity
→ Multi-faceted retrieval mirrors the 7W episodic encoding dimensions.
"""
    },
]

# Primary PDF to auto-download (Generative Agents — most directly relevant)
PRIMARY_PDF = {
    "url":    "https://arxiv.org/pdf/2304.03442",
    "local":  "./generative_agents.pdf",
    "source": "Generative Agents: Interactive Simulacra of Human Behavior — Park et al. (2023)",
    "url_ref":"https://arxiv.org/abs/2304.03442",
}

ALL_REFERENCES = [
    ("Elements of Episodic Memory (Tulving 1983)",    "https://doi.org/10.1093/acprof:oso/9780198521259.001.0001"),
    ("Generative Agents (Park et al., 2023)",          "https://arxiv.org/abs/2304.03442"),
    ("MemGPT (Packer et al., 2023)",                   "https://arxiv.org/abs/2310.08560"),
    ("HippoRAG (Gutierrez et al., 2024)",              "https://arxiv.org/abs/2405.14831"),
    ("A-MEM (Xu et al., 2024)",                        "https://arxiv.org/abs/2312.06681"),
    ("CoALA (Sumers et al., 2023)",                    "https://arxiv.org/abs/2309.02427"),
    ("MemoryBank (Zhong et al., 2023)",                "https://arxiv.org/abs/2305.10250"),
    ("Episodic Memory in Soar (Nuxoll & Laird, 2007)","https://dl.acm.org/doi/10.5555/1597148.1597232"),
    ("Spreading Activation (Collins & Loftus, 1975)", "https://doi.org/10.1037/0033-295X.82.6.407"),
    ("ReAct (Yao et al., 2023)",                       "https://arxiv.org/abs/2210.03629"),
    ("Attention Is All You Need (Vaswani, 2017)",      "https://arxiv.org/abs/1706.03762"),
]

# Pre-built demo episodes for the demo scenario (a software engineer named Alex)
# Represents 2 weeks of interaction history
DEMO_EPISODES_RAW = [
    {
        "what": "Alex introduced himself as a backend engineer with 5 years Python experience",
        "who": ["Alex"], "where": "chat_session_1", "why": "first contact",
        "how": "self-introduction", "emotion_valence": 0.3, "emotion_arousal": 0.2,
        "emotion_label": "positive", "episode_type": "social",
        "importance": 0.9, "tags": ["identity", "python", "backend"],
        "when_offset_days": -14,
    },
    {
        "what": "Alex asked about building a real-time data pipeline using Kafka and Flink",
        "who": ["Alex"], "where": "chat_session_1", "why": "work project requirement",
        "how": "direct question", "emotion_valence": 0.1, "emotion_arousal": 0.4,
        "emotion_label": "neutral", "episode_type": "conversation",
        "importance": 0.8, "tags": ["kafka", "flink", "data_pipeline", "streaming"],
        "when_offset_days": -14,
    },
    {
        "what": "Alex expressed frustration with Kafka consumer lag and offset management",
        "who": ["Alex"], "where": "chat_session_2", "why": "production issue",
        "how": "vent/complaint", "emotion_valence": -0.6, "emotion_arousal": 0.7,
        "emotion_label": "negative", "episode_type": "emotion_event",
        "importance": 0.85, "tags": ["kafka", "bug", "production", "frustration"],
        "when_offset_days": -10,
    },
    {
        "what": "Resolved the Kafka offset issue by resetting consumer group with kafka-consumer-groups.sh",
        "who": ["Alex", "Assistant"], "where": "chat_session_2",
        "why": "fix production outage", "how": "command-line tool debugging",
        "emotion_valence": 0.8, "emotion_arousal": 0.5,
        "emotion_label": "very_positive", "episode_type": "action",
        "importance": 0.95, "tags": ["kafka", "fix", "consumer_group", "resolution"],
        "when_offset_days": -10,
    },
    {
        "what": "Alex mentioned he is preparing for a system design interview next Monday",
        "who": ["Alex"], "where": "chat_session_3", "why": "career goal",
        "how": "casual mention", "emotion_valence": 0.2, "emotion_arousal": 0.6,
        "emotion_label": "neutral", "episode_type": "plan",
        "importance": 0.9, "tags": ["interview", "system_design", "career", "deadline"],
        "when_offset_days": -7,
    },
    {
        "what": "Discussed system design for a URL shortener: hashing, database choice, CDN",
        "who": ["Alex", "Assistant"], "where": "chat_session_3",
        "why": "interview prep", "how": "guided design discussion",
        "emotion_valence": 0.4, "emotion_arousal": 0.5,
        "emotion_label": "positive", "episode_type": "learning",
        "importance": 0.88, "tags": ["system_design", "url_shortener", "database", "hashing"],
        "when_offset_days": -7,
    },
    {
        "what": "Alex decided to use PostgreSQL over MongoDB for his URL shortener project",
        "who": ["Alex"], "where": "chat_session_3",
        "why": "ACID guarantees needed for consistency", "how": "reasoning + comparison",
        "emotion_valence": 0.5, "emotion_arousal": 0.3,
        "emotion_label": "positive", "episode_type": "decision",
        "importance": 0.92, "tags": ["postgresql", "mongodb", "decision", "database"],
        "when_offset_days": -6,
    },
    {
        "what": "Alex asked about vector databases for a semantic search feature he is building",
        "who": ["Alex"], "where": "chat_session_4",
        "why": "new feature requirement", "how": "technical question",
        "emotion_valence": 0.3, "emotion_arousal": 0.4,
        "emotion_label": "neutral", "episode_type": "conversation",
        "importance": 0.82, "tags": ["vector_db", "semantic_search", "embeddings", "rag"],
        "when_offset_days": -4,
    },
    {
        "what": "Explained ChromaDB vs FAISS vs Pinecone trade-offs for production RAG",
        "who": ["Alex", "Assistant"], "where": "chat_session_4",
        "why": "help Alex choose vector DB", "how": "structured comparison",
        "emotion_valence": 0.4, "emotion_arousal": 0.3,
        "emotion_label": "positive", "episode_type": "learning",
        "importance": 0.85, "tags": ["chromadb", "faiss", "pinecone", "rag", "vector_db"],
        "when_offset_days": -4,
    },
    {
        "what": "Alex said he prefers concise technical answers with code examples",
        "who": ["Alex"], "where": "chat_session_4",
        "why": "communication preference", "how": "explicit feedback",
        "emotion_valence": 0.2, "emotion_arousal": 0.2,
        "emotion_label": "neutral", "episode_type": "observation",
        "importance": 0.88, "tags": ["preference", "communication", "code_examples"],
        "when_offset_days": -4,
    },
    {
        "what": "Alex successfully deployed his RAG pipeline to production using ChromaDB",
        "who": ["Alex"], "where": "chat_session_5",
        "why": "feature completion", "how": "Docker + Kubernetes deployment",
        "emotion_valence": 0.9, "emotion_arousal": 0.7,
        "emotion_label": "very_positive", "episode_type": "action",
        "importance": 0.97, "tags": ["rag", "chromadb", "production", "deployment", "success"],
        "when_offset_days": -2,
    },
    {
        "what": "Alex mentioned his team lead praised the RAG system performance",
        "who": ["Alex", "team_lead"], "where": "chat_session_5",
        "why": "positive feedback", "how": "team meeting",
        "emotion_valence": 0.85, "emotion_arousal": 0.6,
        "emotion_label": "very_positive", "episode_type": "social",
        "importance": 0.8, "tags": ["praise", "rag", "team", "recognition"],
        "when_offset_days": -2,
    },
    {
        "what": "Alex is now planning to add re-ranking to his RAG pipeline using cross-encoders",
        "who": ["Alex"], "where": "chat_session_6",
        "why": "improve retrieval quality", "how": "iterative improvement",
        "emotion_valence": 0.4, "emotion_arousal": 0.5,
        "emotion_label": "positive", "episode_type": "plan",
        "importance": 0.85, "tags": ["reranking", "cross_encoder", "rag", "improvement"],
        "when_offset_days": -1,
    },
]


In [9]:
"""
══════════════════════════════════════════════════
Persistence and indexing layer for the episodic memory system.

Three storage backends working in concert:
  1. SQLiteEpisodeStore  — Structured metadata, temporal index, graph edges, full-text
  2. ChromaEpisodeIndex  — Semantic vector embeddings (per-episode + per-chunk)
  3. EpisodeGraph        — In-memory + persistent adjacency map for spreading activation

Implements four of the six retrieval modes:
  semantic    → ChromaEpisodeIndex.semantic_search()
  temporal    → SQLiteEpisodeStore.temporal_query()
  associative → EpisodeGraph.spreading_activation()
  cue_based   → SQLiteEpisodeStore.cue_query()
  (reconstructive and hybrid are handled by RetrievalEngine using these primitives)

Knowledge Base (shared, static):
  KnowledgeBaseStore — ChromaDB, MMR retrieval, built from CORPUS in corpus.py

Reference papers embedded inline:
  • Tulving (1983)         — 7W encoding dimensions
  • Park et al. (2023)    — composite_score formula
  • Collins & Loftus (1975)— spreading activation BFS
  • Nuxoll & Laird (2007) — temporal chaining, interference detection
  • Gutierrez et al. (2024)— graph-based retrieval (HippoRAG)
"""

'\n══════════════════════════════════════════════════\nPersistence and indexing layer for the episodic memory system.\n\nThree storage backends working in concert:\n  1. SQLiteEpisodeStore  — Structured metadata, temporal index, graph edges, full-text\n  2. ChromaEpisodeIndex  — Semantic vector embeddings (per-episode + per-chunk)\n  3. EpisodeGraph        — In-memory + persistent adjacency map for spreading activation\n\nImplements four of the six retrieval modes:\n  semantic    → ChromaEpisodeIndex.semantic_search()\n  temporal    → SQLiteEpisodeStore.temporal_query()\n  associative → EpisodeGraph.spreading_activation()\n  cue_based   → SQLiteEpisodeStore.cue_query()\n  (reconstructive and hybrid are handled by RetrievalEngine using these primitives)\n\nKnowledge Base (shared, static):\n  KnowledgeBaseStore — ChromaDB, MMR retrieval, built from CORPUS in corpus.py\n\nReference papers embedded inline:\n  • Tulving (1983)         — 7W encoding dimensions\n  • Park et al. (2023)    — co

In [17]:

# ══════════════════════════════════════════════════════════════════════
# EMBEDDING MANAGER
# ══════════════════════════════════════════════════════════════════════

class EmbeddingManager:
    _instance: Optional["EmbeddingManager"] = None

    def __init__(self, cfg: Config):
        self.cfg    = cfg
        self._model: Optional[HuggingFaceEmbeddings] = None

    @classmethod
    def get(cls, cfg: Config) -> "EmbeddingManager":
        if cls._instance is None:
            cls._instance = EmbeddingManager(cfg)
        return cls._instance

    def init(self) -> "EmbeddingManager":
        if self._model is None:
            Log.step("Loading HuggingFace Embeddings (local, no API key)")
            Log.info(f"Model : {self.cfg.EMBEDDING_MODEL}")
            Log.info(f"Device: {self.cfg.EMBEDDING_DEVICE}")
            self._model = HuggingFaceEmbeddings(
                model_name=self.cfg.EMBEDDING_MODEL,
                model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
                encode_kwargs={"normalize_embeddings": True, "batch_size": 32},
            )
            Log.ok("Embedding model ready (~90 MB, cached in ~/.cache/huggingface/)")
        return self

    @property
    def model(self) -> HuggingFaceEmbeddings:
        if self._model is None:
            self.init()
        return self._model



In [18]:


# ══════════════════════════════════════════════════════════════════════
# SQLITE EPISODE STORE
# ══════════════════════════════════════════════════════════════════════

class SQLiteEpisodeStore:
    """
    Primary structured store for episode metadata and graph edges.

    Tables:
      episodes    — all 7W dimensions + scoring metadata
      edges       — episode_graph edges with type + weight
      episode_tags — normalized tag-episode mapping

    Supports:
      temporal_query()  — before/after/between timestamp queries
      cue_query()       — multi-facet filter (who, where, emotion, type, tags)
      get_neighbors()   — graph adjacency for spreading activation
    """

    def __init__(self, cfg: Config):
        self.cfg  = cfg
        self._db: Optional[sqlite3.Connection] = None

    def init(self) -> "SQLiteEpisodeStore":
        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)
        self._db = sqlite3.connect(self.cfg.EPISODES_DB, check_same_thread=False)
        self._db.row_factory = sqlite3.Row
        self._db.execute("PRAGMA journal_mode=WAL")
        self._db.execute("PRAGMA foreign_keys=ON")
        self._create_schema()
        n = self._count_all()
        Log.ok(f"SQLiteEpisodeStore: {n} episodes in {self.cfg.EPISODES_DB}")
        return self

    def _create_schema(self):
        cur = self._db.cursor()
        cur.executescript("""
            CREATE TABLE IF NOT EXISTS episodes (
                episode_id       TEXT PRIMARY KEY,
                agent_id         TEXT NOT NULL,
                session_id       TEXT NOT NULL DEFAULT '',
                episode_type     TEXT NOT NULL DEFAULT 'observation',
                what             TEXT NOT NULL,
                when_ts          REAL NOT NULL,
                when_label       TEXT DEFAULT '',
                who_json         TEXT DEFAULT '[]',
                where_ctx        TEXT DEFAULT '',
                why              TEXT DEFAULT '',
                how              TEXT DEFAULT '',
                emotion_valence  REAL DEFAULT 0.0,
                emotion_arousal  REAL DEFAULT 0.0,
                emotion_label    TEXT DEFAULT 'neutral',
                importance       REAL DEFAULT 0.5,
                strength         REAL DEFAULT 1.0,
                access_count     INTEGER DEFAULT 0,
                last_accessed    REAL,
                source           TEXT DEFAULT '',
                linked_ids_json  TEXT DEFAULT '[]',
                raw_text         TEXT DEFAULT '',
                created_at       REAL NOT NULL
            );
            CREATE TABLE IF NOT EXISTS edges (
                edge_id    TEXT PRIMARY KEY,
                source_id  TEXT NOT NULL,
                target_id  TEXT NOT NULL,
                edge_type  TEXT NOT NULL,
                weight     REAL DEFAULT 1.0,
                timestamp  REAL NOT NULL,
                meta_json  TEXT DEFAULT '{}'
            );
            CREATE TABLE IF NOT EXISTS episode_tags (
                episode_id TEXT NOT NULL,
                tag        TEXT NOT NULL,
                PRIMARY KEY (episode_id, tag)
            );
            CREATE INDEX IF NOT EXISTS idx_agent    ON episodes(agent_id);
            CREATE INDEX IF NOT EXISTS idx_when     ON episodes(when_ts);
            CREATE INDEX IF NOT EXISTS idx_type     ON episodes(episode_type);
            CREATE INDEX IF NOT EXISTS idx_emotion  ON episodes(emotion_label);
            CREATE INDEX IF NOT EXISTS idx_src_edge ON edges(source_id);
            CREATE INDEX IF NOT EXISTS idx_tgt_edge ON edges(target_id);
            CREATE INDEX IF NOT EXISTS idx_tag      ON episode_tags(tag);
        """)
        self._db.commit()

    # ── Store ──────────────────────────────────────────────────────────

    def store_episode(self, ep: Episode) -> Episode:
        if not ep.raw_text:
            ep.raw_text = ep.build_raw_text()
        cur = self._db.cursor()
        cur.execute("""
            INSERT OR REPLACE INTO episodes
            (episode_id, agent_id, session_id, episode_type, what, when_ts, when_label,
             who_json, where_ctx, why, how, emotion_valence, emotion_arousal, emotion_label,
             importance, strength, access_count, last_accessed, source, linked_ids_json,
             raw_text, created_at)
            VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
        """, (
            ep.episode_id, ep.agent_id, ep.session_id, ep.episode_type,
            ep.what, ep.when_ts, ep.when_label,
            json.dumps(ep.who), ep.where, ep.why, ep.how,
            ep.emotion_valence, ep.emotion_arousal, ep.emotion_label,
            ep.importance, ep.strength, ep.access_count,
            ep.last_accessed or time.time(), ep.source,
            json.dumps(ep.linked_ids), ep.raw_text, time.time(),
        ))
        # Store tags
        for tag in ep.tags:
            cur.execute(
                "INSERT OR IGNORE INTO episode_tags VALUES (?,?)",
                (ep.episode_id, tag.lower()),
            )
        self._db.commit()
        return ep

    def store_edge(self, edge: EpisodeEdge):
        self._db.execute("""
            INSERT OR REPLACE INTO edges
            (edge_id, source_id, target_id, edge_type, weight, timestamp, meta_json)
            VALUES (?,?,?,?,?,?,?)
        """, (
            edge.edge_id, edge.source_id, edge.target_id,
            edge.edge_type, edge.weight, edge.timestamp,
            json.dumps(edge.metadata),
        ))
        self._db.commit()

    # ── Retrieval ──────────────────────────────────────────────────────

    def get_by_id(self, episode_id: str) -> Optional[Episode]:
        row = self._db.execute(
            "SELECT * FROM episodes WHERE episode_id=?", (episode_id,)
        ).fetchone()
        return self._row_to_episode(row) if row else None

    def get_many(self, episode_ids: List[str]) -> List[Episode]:
        if not episode_ids:
            return []
        placeholders = ",".join("?" * len(episode_ids))
        rows = self._db.execute(
            f"SELECT * FROM episodes WHERE episode_id IN ({placeholders})",
            episode_ids,
        ).fetchall()
        return [self._row_to_episode(r) for r in rows]

    def temporal_query(
        self,
        agent_id: str,
        after_ts: Optional[float] = None,
        before_ts: Optional[float] = None,
        limit: int = 10,
        episode_type: Optional[str] = None,
    ) -> List[Episode]:
        """
        Retrieve episodes within a time window, ordered by recency.
        Implements temporal chaining from Nuxoll & Laird (2007).
        """
        conds = ["agent_id = ?"]
        params: List[Any] = [agent_id]
        if after_ts:
            conds.append("when_ts >= ?"); params.append(after_ts)
        if before_ts:
            conds.append("when_ts <= ?"); params.append(before_ts)
        if episode_type:
            conds.append("episode_type = ?"); params.append(episode_type)
        sql = (
            "SELECT * FROM episodes WHERE " + " AND ".join(conds)
            + " ORDER BY when_ts DESC LIMIT ?"
        )
        params.append(limit)
        rows = self._db.execute(sql, params).fetchall()
        results = [self._row_to_episode(r) for r in rows]
        Log.mode("temporal", f"[{agent_id}] window query → {len(results)} episodes")
        return results

    def cue_query(
        self,
        agent_id: str,
        who: Optional[List[str]] = None,
        where: Optional[str] = None,
        emotion_label: Optional[str] = None,
        emotion_valence_min: Optional[float] = None,
        emotion_valence_max: Optional[float] = None,
        episode_type: Optional[str] = None,
        tags: Optional[List[str]] = None,
        limit: int = 10,
    ) -> List[Episode]:
        """
        Multi-facet cue-based retrieval: filter by any combination of 7W dimensions.
        Models the encoding specificity principle (Tulving, 1983):
        retrieval cues that match encoding context produce better recall.
        """
        conds = ["e.agent_id = ?"]
        params: List[Any] = [agent_id]

        if where:
            conds.append("e.where_ctx LIKE ?"); params.append(f"%{where}%")
        if emotion_label:
            conds.append("e.emotion_label = ?"); params.append(emotion_label)
        if emotion_valence_min is not None:
            conds.append("e.emotion_valence >= ?"); params.append(emotion_valence_min)
        if emotion_valence_max is not None:
            conds.append("e.emotion_valence <= ?"); params.append(emotion_valence_max)
        if episode_type:
            conds.append("e.episode_type = ?"); params.append(episode_type)

        base_sql = (
            "SELECT DISTINCT e.* FROM episodes e "
        )
        # Tag join
        if tags:
            for i, tag in enumerate(tags):
                alias = f"t{i}"
                base_sql += (
                    f"JOIN episode_tags {alias} "
                    f"ON e.episode_id={alias}.episode_id AND {alias}.tag=? "
                )
                params.append(tag.lower())

        base_sql += "WHERE " + " AND ".join(conds)
        base_sql += " ORDER BY e.when_ts DESC LIMIT ?"
        params.append(limit)

        rows = self._db.execute(base_sql, params).fetchall()
        results = [self._row_to_episode(r) for r in rows]

        # WHO filter (post-processing — stored as JSON array)
        if who:
            who_lower = [w.lower() for w in who]
            results = [
                r for r in results
                if any(p.lower() in who_lower for p in r.who)
            ]

        Log.mode("cue_based", f"[{agent_id}] facet query → {len(results)} episodes")
        return results

    def get_neighbors(self, episode_id: str) -> List[Tuple[str, str, float]]:
        """Return (neighbor_id, edge_type, weight) for all edges from episode_id."""
        rows = self._db.execute(
            "SELECT target_id, edge_type, weight FROM edges WHERE source_id=? "
            "UNION SELECT source_id, edge_type, weight FROM edges WHERE target_id=?",
            (episode_id, episode_id),
        ).fetchall()
        return [(r["target_id"] if r["target_id"] != episode_id else r["source_id"],
                 r["edge_type"], r["weight"]) for r in rows]

    def get_all_for_agent(self, agent_id: str) -> List[Episode]:
        rows = self._db.execute(
            "SELECT * FROM episodes WHERE agent_id=? ORDER BY when_ts DESC",
            (agent_id,),
        ).fetchall()
        return [self._row_to_episode(r) for r in rows]

    def mark_accessed(self, episode_id: str):
        self._db.execute("""
            UPDATE episodes
            SET access_count = access_count + 1,
                last_accessed = ?
            WHERE episode_id = ?
        """, (time.time(), episode_id))
        self._db.commit()

    def _count_all(self) -> int:
        return self._db.execute("SELECT COUNT(*) FROM episodes").fetchone()[0]

    def count_for_agent(self, agent_id: str) -> int:
        return self._db.execute(
            "SELECT COUNT(*) FROM episodes WHERE agent_id=?", (agent_id,)
        ).fetchone()[0]

    def _row_to_episode(self, row: sqlite3.Row) -> Episode:
        ep = Episode(
            episode_id=row["episode_id"],
            agent_id=row["agent_id"],
            session_id=row["session_id"],
            episode_type=row["episode_type"],
            what=row["what"],
            when_ts=row["when_ts"],
            when_label=row["when_label"] or "",
            who=json.loads(row["who_json"]),
            where=row["where_ctx"] or "",
            why=row["why"] or "",
            how=row["how"] or "",
            emotion_valence=float(row["emotion_valence"]),
            emotion_arousal=float(row["emotion_arousal"]),
            emotion_label=row["emotion_label"],
            importance=float(row["importance"]),
            strength=float(row["strength"]),
            access_count=int(row["access_count"]),
            last_accessed=float(row["last_accessed"]) if row["last_accessed"] else time.time(),
            source=row["source"] or "",
            linked_ids=json.loads(row["linked_ids_json"]),
            raw_text=row["raw_text"] or "",
        )
        # Reload tags
        tags = self._db.execute(
            "SELECT tag FROM episode_tags WHERE episode_id=?", (ep.episode_id,)
        ).fetchall()
        ep.tags = [r["tag"] for r in tags]
        return ep


In [19]:

# ══════════════════════════════════════════════════════════════════════
# EPISODE GRAPH — Spreading Activation
# ══════════════════════════════════════════════════════════════════════

class EpisodeGraph:
    """
    In-memory episode graph for spreading activation retrieval.
    Implements Collins & Loftus (1975) spreading activation theory.

    Nodes: episode_ids
    Edges: typed connections (follows, causes, contradicts, supports, ...)
    Activation propagates via BFS with exponential decay per hop.

    activation(neighbor) = activation(source) × ACTIVATION_DECAY
    Nodes below MIN_ACTIVATION threshold are pruned from results.
    """

    def __init__(self, cfg: Config, sql_store: SQLiteEpisodeStore):
        self.cfg       = cfg
        self.sql       = sql_store
        # adjacency: episode_id → list of (neighbor_id, edge_type, weight)
        self._adj: Dict[str, List[Tuple[str, str, float]]] = defaultdict(list)
        self._loaded: bool = False

    def build_from_store(self, agent_id: str):
        """Load all edges for an agent into the in-memory adjacency map."""
        episodes = self.sql.get_all_for_agent(agent_id)
        self._adj.clear()
        for ep in episodes:
            neighbors = self.sql.get_neighbors(ep.episode_id)
            for neighbor_id, edge_type, weight in neighbors:
                # Add bidirectional for undirected edges (similar_to, involves)
                self._adj[ep.episode_id].append((neighbor_id, edge_type, weight))
        self._loaded = True
        Log.mode(
            "associative",
            f"[{agent_id}] graph built — {len(self._adj)} nodes, "
            f"{sum(len(v) for v in self._adj.values())} edges"
        )

    def add_edge(self, edge: EpisodeEdge):
        """Add edge to in-memory graph."""
        self._adj[edge.source_id].append(
            (edge.target_id, edge.edge_type, edge.weight)
        )
        # Undirected edge types — add reverse
        if edge.edge_type in (EdgeType.SIMILAR_TO.value, EdgeType.INVOLVES.value,
                              EdgeType.SUPPORTS.value, EdgeType.CONTRADICTS.value):
            self._adj[edge.target_id].append(
                (edge.source_id, edge.edge_type, edge.weight)
            )

    def spreading_activation(
        self,
        seed_ids: List[str],
        seed_activations: Optional[Dict[str, float]] = None,
        depth: Optional[int] = None,
        decay: Optional[float] = None,
        threshold: Optional[float] = None,
        excluded_types: Optional[Set[str]] = None,
    ) -> Dict[str, float]:
        """
        BFS spreading activation from seed episodes.
        Returns {episode_id: activation_score} for all reached nodes.

        Implements Collins & Loftus (1975):
          A(neighbor) = A(source) × decay_factor × edge_weight
        """
        max_depth  = depth    or self.cfg.ACTIVATION_DEPTH
        dec_factor = decay    or self.cfg.ACTIVATION_DECAY
        min_act    = threshold or self.cfg.MIN_ACTIVATION
        excl       = excluded_types or set()

        # Initialize activation map
        activation: Dict[str, float] = {}
        if seed_activations:
            activation.update(seed_activations)
        else:
            for sid in seed_ids:
                activation[sid] = 1.0

        # BFS queue: (episode_id, current_depth)
        queue: deque = deque()
        for sid in seed_ids:
            queue.append((sid, 0))
        visited: Set[str] = set(seed_ids)

        hops = 0
        while queue:
            node_id, depth_now = queue.popleft()
            if depth_now >= max_depth:
                continue
            node_act = activation.get(node_id, 0.0)

            for neighbor_id, edge_type, weight in self._adj.get(node_id, []):
                if edge_type in excl:
                    continue
                new_act = node_act * dec_factor * weight
                if new_act < min_act:
                    continue
                # Accumulate activation (multiple paths to same node add up)
                activation[neighbor_id] = activation.get(neighbor_id, 0.0) + new_act
                if neighbor_id not in visited:
                    visited.add(neighbor_id)
                    queue.append((neighbor_id, depth_now + 1))
                    hops += 1

        # Remove seeds and prune below threshold
        result = {
            k: v for k, v in activation.items()
            if k not in set(seed_ids) and v >= min_act
        }
        Log.mode(
            "associative",
            f"activation from {len(seed_ids)} seeds → "
            f"{len(result)} activated nodes (depth={max_depth}, hops={hops})"
        )
        return result

    def find_contradictions(self, episode_id: str) -> List[Tuple[str, float]]:
        """Return episodes connected by CONTRADICTS edges."""
        return [
            (nid, w)
            for nid, etype, w in self._adj.get(episode_id, [])
            if etype == EdgeType.CONTRADICTS.value
        ]

    def get_causal_chain(
        self, episode_id: str, direction: str = "forward"
    ) -> List[str]:
        """
        Traverse CAUSES / TRIGGERED_BY edges to build a causal chain.
        direction: 'forward' = what this caused, 'backward' = what caused this
        """
        chain: List[str] = []
        current = episode_id
        visited: Set[str] = set()
        while True:
            if current in visited:
                break
            visited.add(current)
            next_node = None
            for nid, etype, _ in self._adj.get(current, []):
                if direction == "forward" and etype == EdgeType.CAUSES.value:
                    next_node = nid; break
                elif direction == "backward" and etype == EdgeType.TRIGGERED_BY.value:
                    next_node = nid; break
            if next_node is None:
                break
            chain.append(next_node)
            current = next_node
        return chain


In [20]:

# ══════════════════════════════════════════════════════════════════════
# CHROMA EPISODE INDEX — Semantic Similarity
# ══════════════════════════════════════════════════════════════════════

class ChromaEpisodeIndex:
    """
    Semantic vector index over episodes.
    Each episode's raw_text (7W concatenated) is embedded and stored.
    Supports MMR retrieval for diversity + composite re-ranking.
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg        = cfg
        self.em         = em
        self._client: Optional[chromadb.PersistentClient] = None
        self._coll: Optional[Any] = None

    def init(self) -> "ChromaEpisodeIndex":
        Path(self.cfg.CHROMA_DIR).mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        self._coll = self._client.get_or_create_collection(
            name=self.cfg.CHROMA_COLLECTION,
            metadata={"hnsw:space": "cosine"},
        )
        n = self._coll.count()
        Log.ok(f"ChromaEpisodeIndex: {n} vectors in {self.cfg.CHROMA_COLLECTION}")
        return self

    def index_episode(self, ep: Episode):
        """Upsert an episode's embedding into ChromaDB."""
        raw = ep.raw_text or ep.build_raw_text()
        meta = {
            "agent_id":       ep.agent_id,
            "episode_type":   ep.episode_type,
            "importance":     ep.importance,
            "when_ts":        ep.when_ts,
            "emotion_label":  ep.emotion_label,
            "emotion_valence":ep.emotion_valence,
        }
        try:
            self._coll.upsert(
                ids=[ep.episode_id],
                documents=[raw],
                metadatas=[meta],
            )
        except Exception as exc:
            Log.warn(f"ChromaEpisodeIndex.index failed: {exc}")

    def semantic_search(
        self,
        agent_id: str,
        query: str,
        k: int = 5,
        filter_type: Optional[str] = None,
    ) -> List[Tuple[str, float]]:
        """
        Semantic search returning (episode_id, cosine_similarity) pairs.
        Applies agent_id isolation and optional episode_type filter.
        Returns list sorted by similarity descending.
        """
        n_total = self._coll.count()
        if n_total == 0:
            return []

        where: Dict[str, Any] = {"agent_id": {"$eq": agent_id}}
        if filter_type:
            where = {
                "$and": [
                    {"agent_id": {"$eq": agent_id}},
                    {"episode_type": {"$eq": filter_type}},
                ]
            }

        try:
            res = self._coll.query(
                query_texts=[query],
                n_results=min(k * 2, n_total),
                where=where,
            )
        except Exception as exc:
            Log.warn(f"ChromaEpisodeIndex.search failed: {exc}")
            return []

        if not res["ids"] or not res["ids"][0]:
            return []

        results: List[Tuple[str, float]] = []
        for ep_id, dist in zip(res["ids"][0], res["distances"][0]):
            sim = max(0.0, 1.0 - dist)   # cosine distance → similarity
            results.append((ep_id, sim))

        results.sort(key=lambda x: x[1], reverse=True)
        Log.mode("semantic", f"[{agent_id}] '{query[:50]}' → {len(results[:k])} hits")
        return results[:k]

    def detect_interference(
        self,
        episode_id: str,
        agent_id: str,
        threshold: Optional[float] = None,
    ) -> List[Tuple[str, float]]:
        """
        Find episodes very similar to the given one (potential interference).
        Nuxoll & Laird (2007): two similar cues → retrieval competition.
        Returns (conflicting_episode_id, similarity) pairs above threshold.
        """
        thr = threshold or self.cfg.INTERFERENCE_THRESHOLD
        res = self._coll.get(ids=[episode_id], include=["documents"])
        if not res["documents"]:
            return []
        doc_text = res["documents"][0]

        n_total = self._coll.count()
        if n_total < 2:
            return []

        raw = self._coll.query(
            query_texts=[doc_text],
            n_results=min(6, n_total),
            where={"agent_id": {"$eq": agent_id}},
        )
        conflicts: List[Tuple[str, float]] = []
        for eid, dist in zip(raw["ids"][0], raw["distances"][0]):
            sim = max(0.0, 1.0 - dist)
            if eid != episode_id and sim >= thr:
                conflicts.append((eid, sim))
        return conflicts



In [21]:


# ══════════════════════════════════════════════════════════════════════
# KNOWLEDGE BASE STORE
# ══════════════════════════════════════════════════════════════════════

class KnowledgeBaseStore:
    """Static shared knowledge base (10 papers) — MMR retrieval."""

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg     = cfg
        self.em      = em
        self._store: Optional[Chroma] = None
        self._splitter = RecursiveCharacterTextSplitter(
            chunk_size=cfg.CHUNK_SIZE,
            chunk_overlap=cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " "],
        )

    def build(self) -> "KnowledgeBaseStore":
        Log.step("Building Knowledge Base (11 reference papers)")
        Path(self.cfg.KB_CHROMA_DIR).mkdir(parents=True, exist_ok=True)

        # Attempt PDF download (Generative Agents paper)
        raw: List[Document] = self._try_pdf()

        # Inline corpus
        for entry in CORPUS:
            raw.append(Document(
                page_content=entry["content"].strip(),
                metadata={"source": entry["source"], "url": entry["url"],
                          "doc_id": entry["id"]},
            ))

        # Chunk + deduplicate
        chunks = self._splitter.split_documents(raw)
        seen: set = set()
        unique: List[Document] = []
        for i, c in enumerate(chunks):
            h = hashlib.md5(c.page_content.encode()).hexdigest()[:10]
            if h not in seen:
                seen.add(h)
                c.metadata["chunk_id"] = i
                unique.append(c)

        # Build ChromaDB collection
        client = chromadb.PersistentClient(
            path=self.cfg.KB_CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        try:
            client.delete_collection(self.cfg.KB_COLLECTION)
        except Exception:
            pass

        self._store = Chroma.from_documents(
            documents=unique,
            embedding=self.em.model,
            client=client,
            collection_name=self.cfg.KB_COLLECTION,
            collection_metadata={"hnsw:space": "cosine"},
        )
        n = self._store._collection.count()
        Log.ok(f"KB: {n} chunks indexed")
        return self

    def _try_pdf(self) -> List[Document]:
        try:
            import urllib.request
            from langchain_community.document_loaders import PyPDFLoader
            if not Path(PRIMARY_PDF["local"]).exists():
                Log.info(f"Downloading Generative Agents PDF…")
                urllib.request.urlretrieve(PRIMARY_PDF["url"], PRIMARY_PDF["local"])
                Log.ok(f"PDF saved → {PRIMARY_PDF['local']}")
            pages = PyPDFLoader(PRIMARY_PDF["local"]).load()
            for p in pages:
                p.metadata.update({"source": PRIMARY_PDF["source"],
                                   "url": PRIMARY_PDF["url_ref"]})
            Log.ok(f"PDF loaded: {len(pages)} pages")
            return pages
        except Exception as exc:
            Log.warn(f"PDF skip ({exc}), using inline corpus only")
            return []

    def retrieve(self, query: str, k: int = 3) -> List[Document]:
        retriever = self._store.as_retriever(
            search_type="mmr",
            search_kwargs={"k": k, "fetch_k": self.cfg.FETCH_K,
                           "lambda_mult": self.cfg.MMR_LAMBDA},
        )
        return retriever.invoke(query)


# ══════════════════════════════════════════════════════════════════════
# EPISODE FACTORY — build episodes from raw dicts
# ══════════════════════════════════════════════════════════════════════

def build_episode_from_dict(
    agent_id: str,
    d: Dict[str, Any],
    session_id: str = "demo_session",
) -> Episode:
    """
    Construct an Episode from a raw dict (e.g. from DEMO_EPISODES_RAW).
    Handles time offset (when_offset_days → absolute timestamp).
    """
    offset_days = d.get("when_offset_days", 0)
    when_ts     = time.time() + (offset_days * 86400.0)

    age_days = abs(offset_days)
    when_label = (
        f"{age_days}d ago" if age_days > 0
        else "today"
    )

    ep = Episode(
        episode_id   = f"ep_{agent_id[:4]}_{str(uuid.uuid4())[:8]}",
        agent_id     = agent_id,
        session_id   = session_id,
        what         = d.get("what", ""),
        when_ts      = when_ts,
        when_label   = when_label,
        who          = d.get("who", []),
        where        = d.get("where", ""),
        why          = d.get("why", ""),
        how          = d.get("how", ""),
        emotion_valence = float(d.get("emotion_valence", 0.0)),
        emotion_arousal = float(d.get("emotion_arousal", 0.0)),
        emotion_label   = d.get("emotion_label", EmotionLabel.NEUTRAL.value),
        episode_type    = d.get("episode_type", EpisodeType.OBSERVATION.value),
        importance      = float(d.get("importance", 0.5)),
        strength        = 1.0,
        tags            = d.get("tags", []),
        source          = d.get("source", "demo"),
    )
    ep.raw_text = ep.build_raw_text()
    return ep


def auto_link_episodes(
    episodes: List[Episode],
    sql_store: SQLiteEpisodeStore,
    graph: EpisodeGraph,
) -> int:
    """
    Automatically create graph edges between episodes:
      FOLLOWS   — episodes in same session, sequential order
      INVOLVES  — episodes sharing a participant (WHO overlap)
      CAUSES    — decision episodes that follow emotion_event episodes

    Returns number of edges created.
    """
    edge_count = 0

    # Sort by timestamp
    sorted_eps = sorted(episodes, key=lambda e: e.when_ts)

    for i, ep in enumerate(sorted_eps):
        # FOLLOWS — next episode in same session
        if i + 1 < len(sorted_eps):
            next_ep = sorted_eps[i + 1]
            if ep.session_id == next_ep.session_id:
                edge = EpisodeEdge(
                    edge_id   = f"edge_{str(uuid.uuid4())[:8]}",
                    source_id = ep.episode_id,
                    target_id = next_ep.episode_id,
                    edge_type = EdgeType.FOLLOWS.value,
                    weight    = 1.0,
                )
                sql_store.store_edge(edge)
                graph.add_edge(edge)
                edge_count += 1

        # INVOLVES — shared participants
        for j in range(i + 1, min(i + 6, len(sorted_eps))):
            other = sorted_eps[j]
            shared = set(ep.who) & set(other.who)
            if shared and ep.episode_id != other.episode_id:
                edge = EpisodeEdge(
                    edge_id   = f"edge_{str(uuid.uuid4())[:8]}",
                    source_id = ep.episode_id,
                    target_id = other.episode_id,
                    edge_type = EdgeType.INVOLVES.value,
                    weight    = len(shared) / max(len(ep.who), len(other.who), 1),
                )
                sql_store.store_edge(edge)
                graph.add_edge(edge)
                edge_count += 1
                break   # one INVOLVES edge per episode pair is enough

        # CAUSES — emotion_event followed closely by decision
        if (ep.episode_type == EpisodeType.EMOTION_EVENT.value
                and i + 1 < len(sorted_eps)):
            next_ep = sorted_eps[i + 1]
            if next_ep.episode_type in (
                EpisodeType.DECISION.value, EpisodeType.ACTION.value
            ):
                edge = EpisodeEdge(
                    edge_id   = f"edge_{str(uuid.uuid4())[:8]}",
                    source_id = ep.episode_id,
                    target_id = next_ep.episode_id,
                    edge_type = EdgeType.CAUSES.value,
                    weight    = abs(ep.emotion_valence),
                )
                sql_store.store_edge(edge)
                graph.add_edge(edge)
                edge_count += 1

    return edge_count


In [15]:
"""
═════════════════════════════════════════════════════
Orchestrates all six episodic retrieval modes:

  1. semantic       — embedding similarity (all-MiniLM-L6-v2)
  2. temporal       — time-window / before / after / during queries
  3. associative    — spreading activation through episode graph
  4. cue_based      — multi-facet filter (WHO, WHERE, EMOTION, TYPE, TAGS)
  5. reconstructive — LLM rebuilds full episode from retrieved fragments
  6. hybrid         — all modes fused with composite re-ranking

Composite re-ranking (Park et al., 2023):
  score = (W_rec × recency + W_imp × importance + W_rel × relevance) / ΣW
  All three components are normalized to [0, 1].

LLM Prompts:
  IMPORTANCE_PROMPT     — rate episode importance 0–1
  STORE_PROMPT          — extract 7W dimensions from free text
  RECONSTRUCT_PROMPT    — rebuild full episode from fragments
  NARRATIVE_PROMPT      — generate episodic narrative summary
  ANSWER_PROMPT         — answer query grounded in retrieved episodes
  INTERFERENCE_PROMPT   — identify which conflicting episode is more reliable
"""

'\n═════════════════════════════════════════════════════\nOrchestrates all six episodic retrieval modes:\n\n  1. semantic       — embedding similarity (all-MiniLM-L6-v2)\n  2. temporal       — time-window / before / after / during queries\n  3. associative    — spreading activation through episode graph\n  4. cue_based      — multi-facet filter (WHO, WHERE, EMOTION, TYPE, TAGS)\n  5. reconstructive — LLM rebuilds full episode from retrieved fragments\n  6. hybrid         — all modes fused with composite re-ranking\n\nComposite re-ranking (Park et al., 2023):\n  score = (W_rec × recency + W_imp × importance + W_rel × relevance) / ΣW\n  All three components are normalized to [0, 1].\n\nLLM Prompts:\n  IMPORTANCE_PROMPT     — rate episode importance 0–1\n  STORE_PROMPT          — extract 7W dimensions from free text\n  RECONSTRUCT_PROMPT    — rebuild full episode from fragments\n  NARRATIVE_PROMPT      — generate episodic narrative summary\n  ANSWER_PROMPT         — answer query grounded 

In [16]:

# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

IMPORTANCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Rate how important this event is for long-term episodic memory, 0.0–1.0.\n"
     "High (0.8–1.0): decisions, resolutions, emotional peaks, new facts about the agent.\n"
     "Medium (0.4–0.7): useful exchanges, learning, plans.\n"
     "Low (0.0–0.3): greetings, repetitions, trivial chitchat.\n"
     "Respond ONLY with JSON: {\"score\": 0.0-1.0, \"reason\": \"brief reason\"}"),
    ("human", "Event: {event}"),
])

STORE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Extract the 7W episodic dimensions from this text.\n"
     "Respond ONLY with valid JSON (no markdown fences):\n"
     "{\n"
     "  \"what\": \"core action/event\",\n"
     "  \"who\": [\"participant1\", \"participant2\"],\n"
     "  \"where\": \"context/location\",\n"
     "  \"why\": \"motivation/cause\",\n"
     "  \"how\": \"method/manner\",\n"
     "  \"when_label\": \"temporal label (e.g. 'this morning')\",\n"
     "  \"episode_type\": \"observation|action|conversation|decision|learning|"
     "emotion_event|social|reflection|plan|correction\",\n"
     "  \"emotion_valence\": -1.0 to 1.0,\n"
     "  \"emotion_arousal\": 0.0 to 1.0,\n"
     "  \"emotion_label\": \"very_positive|positive|neutral|negative|very_negative\",\n"
     "  \"importance\": 0.0 to 1.0,\n"
     "  \"tags\": [\"tag1\", \"tag2\"]\n"
     "}"),
    ("human", "Text to parse: {text}"),
])

RECONSTRUCT_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are reconstructing an episodic memory from fragments.\n"
     "Human episodic memory is reconstructive — we rebuild episodes from partial cues.\n"
     "Use the retrieved episode fragments to reconstruct the most complete, coherent "
     "account of the relevant experience. Fill in likely details where fragments are "
     "incomplete, but clearly mark inferred details with [inferred].\n"
     "Structure your reconstruction as:\n"
     "WHAT HAPPENED: ...\n"
     "CONTEXT (WHO/WHERE): ...\n"
     "WHY IT HAPPENED: ...\n"
     "EMOTIONAL TONE: ...\n"
     "KEY DETAILS: ...\n"
     "CONFIDENCE: [high/medium/low]"),
    ("human",
     "Query: {query}\n\nRetrieved fragments:\n{fragments}\n\nReconstruct the memory:"),
])

NARRATIVE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Generate a coherent episodic narrative from these memory episodes.\n"
     "Write in second person ('You...') to create an immersive recall experience.\n"
     "Preserve temporal order. Note emotional arcs and significant turning points.\n"
     "Keep the narrative concise (3–5 sentences) but vivid and specific."),
    ("human", "Episodes (chronological):\n{episodes}\n\nGenerate the narrative memory:"),
])

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are an episodic memory retrieval system answering a query using retrieved memories.\n\n"
     "Rules:\n"
     "1. Ground your answer ONLY in the retrieved episodes and knowledge base.\n"
     "2. Cite episodes by their type and approximate age (e.g. '[DECISION, 6d ago]').\n"
     "3. If the reconstructed memory is available, use it for richer detail.\n"
     "4. Note any conflicts/interference between episodes if present.\n"
     "5. If episodic evidence is insufficient, say so and use the knowledge base.\n"
     "6. Match the agent's known preferences (if in their profile).\n\n"
     "Retrieved episodic context:\n{episodic_context}\n\n"
     "Knowledge base context:\n{kb_context}\n\n"
     "Reconstructed memory:\n{reconstructed}"),
    ("human", "{query}"),
])

INTERFERENCE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Two similar episodic memories may be interfering. Determine which is more reliable.\n"
     "Consider: recency, specificity, emotional salience, coherence with known facts.\n"
     "Respond ONLY with JSON: "
     "{\"preferred_id\": \"episode_id\", \"reason\": \"brief reason\", "
     "\"confidence\": 0.0-1.0}"),
    ("human",
     "Episode A (id={id_a}):\n{ep_a}\n\n"
     "Episode B (id={id_b}):\n{ep_b}\n\n"
     "Query: {query}\n\nWhich is more reliable?"),
])


In [22]:

# ══════════════════════════════════════════════════════════════════════
# RETRIEVAL ENGINE
# ══════════════════════════════════════════════════════════════════════

class EpisodicRetrievalEngine:
    """
    Unified episodic memory retrieval engine.

    Implements all 6 retrieval modes and the full storage pipeline.
    All retrieval uses the composite_score formula from Park et al. (2023).
    """

    def __init__(self, cfg: Config):
        self.cfg     = cfg
        self._parser = StrOutputParser()

        # Components (initialized in setup())
        self.em:      Optional[EmbeddingManager]   = None
        self.sql:     Optional[SQLiteEpisodeStore] = None
        self.graph:   Optional[EpisodeGraph]       = None
        self.index:   Optional[ChromaEpisodeIndex] = None
        self.kb:      Optional[KnowledgeBaseStore] = None
        self.llm:     Optional[ChatOllama]    = None

    # ── Setup ──────────────────────────────────────────────────────────

    def setup(self) -> "EpisodicRetrievalEngine":
        Log.banner(
            "EPISODIC MEMORY RETRIEVAL RAG",
            "LangChain · Azure OpenAI · HuggingFace · ChromaDB · SQLite · Episode Graph"
        )
        from pathlib import Path
        Path(self.cfg.BASE_DIR).mkdir(parents=True, exist_ok=True)

        # Embeddings
        self.em = EmbeddingManager.get(self.cfg)
        self.em.init()

        # SQLite store
        self.sql = SQLiteEpisodeStore(self.cfg)
        self.sql.init()

        # Episode graph (in-memory, loaded from SQL)
        self.graph = EpisodeGraph(self.cfg, self.sql)

        # Chroma semantic index
        self.index = ChromaEpisodeIndex(self.cfg, self.em)
        self.index.init()

        # Knowledge base
        self.kb = KnowledgeBaseStore(self.cfg, self.em)
        self.kb.build()

        # LLM
        if self.cfg.is_configured():
            Log.step("Connecting to Azure OpenAI")
            self.llm = ChatOllama(
                base_url=getattr(config, 'OLLAMA_BASE_URL', 'http://localhost:11434'),
                model=getattr(config, 'OLLAMA_MODEL', 'qwen2.5-coder:7b'),
                temperature=getattr(config, 'LLM_TEMPERATURE', 0.0),
                num_predict=getattr(config, 'LLM_MAX_TOKENS', 512),
            )
            Log.ok(f"LLM ready — {self.cfg.OLLAMA_MODEL}")
        else:
            Log.warn("Azure OpenAI not configured — LLM calls will fail gracefully")

        Log.banner("SYSTEM READY — all six retrieval modes available")
        return self

    def _invoke(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        if self.llm is None:
            return "[LLM not configured — set OLLAMA_BASE_URL + OLLAMA_BASE_URL]"
        try:
            chain = prompt | self.llm | self._parser
            return chain.invoke(kwargs)
        except Exception as exc:
            Log.err(f"LLM: {exc}")
            return f"[LLM error: {exc}]"

    def _parse_json(self, raw: str, fallback: Any = None) -> Any:
        try:
            clean = raw.strip().lstrip("```json").lstrip("```").rstrip("```").strip()
            return json.loads(clean)
        except json.JSONDecodeError:
            return fallback if fallback is not None else {}

    # ── Episode Storage Pipeline ───────────────────────────────────────

    def store_episode(
        self,
        agent_id: str,
        text: str,
        session_id: str = "",
        manual_overrides: Optional[Dict[str, Any]] = None,
    ) -> Episode:
        """
        Full storage pipeline:
          1. LLM extracts 7W dimensions from free text
          2. Importance is scored
          3. Episode stored in SQLite + ChromaDB
          4. Graph edges auto-linked to nearby episodes
        """
        Log.step(f"Storing episode for [{agent_id}]: {text[:60]}…")

        # 1. Extract 7W from LLM
        raw = self._invoke(STORE_PROMPT, text=text)
        dims = self._parse_json(raw, {})

        # 2. Merge with overrides
        if manual_overrides:
            dims.update(manual_overrides)

        # 3. Build Episode
        ep = Episode(
            episode_id      = f"ep_{agent_id[:4]}_{str(uuid.uuid4())[:8]}",
            agent_id        = agent_id,
            session_id      = session_id or str(uuid.uuid4())[:8],
            what            = dims.get("what", text[:200]),
            when_ts         = time.time(),
            when_label      = dims.get("when_label", "just now"),
            who             = dims.get("who", [agent_id]),
            where           = dims.get("where", ""),
            why             = dims.get("why", ""),
            how             = dims.get("how", ""),
            emotion_valence = float(dims.get("emotion_valence", 0.0)),
            emotion_arousal = float(dims.get("emotion_arousal", 0.0)),
            emotion_label   = dims.get("emotion_label", EmotionLabel.NEUTRAL.value),
            episode_type    = dims.get("episode_type", EpisodeType.OBSERVATION.value),
            importance      = float(dims.get("importance", 0.5)),
            tags            = dims.get("tags", []),
            source          = "llm_extraction",
        )
        ep.raw_text = ep.build_raw_text()

        # 4. Store
        self.sql.store_episode(ep)
        self.index.index_episode(ep)

        # 5. Auto-link to recent episodes in same session
        recent = self.sql.temporal_query(
            agent_id, after_ts=time.time() - 3600, limit=5
        )
        if recent:
            auto_link_episodes(recent + [ep], self.sql, self.graph)

        Log.ok(f"Episode stored: [{ep.episode_type}] {ep.what[:60]}")
        return ep

    def load_demo_episodes(self, agent_id: str, session_id: str = "demo") -> int:
        """
        Load all DEMO_EPISODES_RAW for an agent. Used for demo/testing.
        Builds graph edges after loading.
        """
        from corpus import DEMO_EPISODES_RAW
        Log.step(f"Loading {len(DEMO_EPISODES_RAW)} demo episodes for [{agent_id}]")

        episodes: List[Episode] = []
        for raw_dict in DEMO_EPISODES_RAW:
            ep = build_episode_from_dict(agent_id, raw_dict, session_id=session_id)
            self.sql.store_episode(ep)
            self.index.index_episode(ep)
            episodes.append(ep)
            Log.info(f"  [{ep.episode_type:15s}] {ep.what[:60]}")

        # Build episode graph
        n_edges = auto_link_episodes(episodes, self.sql, self.graph)
        self.graph.build_from_store(agent_id)

        Log.ok(f"Loaded {len(episodes)} episodes, {n_edges} graph edges for [{agent_id}]")
        return len(episodes)

    # ── Composite Re-ranking ───────────────────────────────────────────

    def _rerank(
        self,
        episodes: List[Episode],
        relevance_map: Optional[Dict[str, float]] = None,
    ) -> List[Tuple[Episode, float]]:
        """
        Apply composite_score (Park et al., 2023) to a list of episodes.
        Returns sorted (episode, score) pairs descending.
        """
        relevance_map = relevance_map or {}
        scored: List[Tuple[Episode, float]] = []
        for ep in episodes:
            rel  = relevance_map.get(ep.episode_id, 0.5)
            score = ep.composite_score(
                relevance=rel,
                w_recency=self.cfg.W_RECENCY,
                w_importance=self.cfg.W_IMPORTANCE,
                w_relevance=self.cfg.W_RELEVANCE,
                decay_base=self.cfg.RECENCY_DECAY_BASE,
            )
            scored.append((ep, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    # ── Mode 1: Semantic Retrieval ─────────────────────────────────────

    def retrieve_semantic(
        self, agent_id: str, query: str, k: Optional[int] = None
    ) -> List[Tuple[Episode, float]]:
        """
        Embedding similarity search over all episodes.
        Re-ranked using composite_score.
        """
        k = k or self.cfg.TOP_K_SEMANTIC
        hits = self.index.semantic_search(agent_id, query, k=k)

        episode_ids = [eid for eid, _ in hits]
        relevance   = {eid: sim for eid, sim in hits}
        episodes    = self.sql.get_many(episode_ids)
        for ep in episodes:
            self.sql.mark_accessed(ep.episode_id)

        return self._rerank(episodes, relevance)

    # ── Mode 2: Temporal Retrieval ─────────────────────────────────────

    def retrieve_temporal(
        self,
        agent_id: str,
        query: str,
        after_ts: Optional[float] = None,
        before_ts: Optional[float] = None,
        days_back: Optional[float] = None,
        episode_type: Optional[str] = None,
        k: Optional[int] = None,
    ) -> List[Tuple[Episode, float]]:
        """
        Time-window retrieval. Supports:
          • absolute timestamps (after_ts / before_ts)
          • relative window (days_back=7 → last 7 days)
        Implements temporal chaining (Nuxoll & Laird, 2007).
        """
        k = k or self.cfg.TOP_K_TEMPORAL

        if days_back is not None:
            after_ts = time.time() - days_back * 86400.0

        episodes = self.sql.temporal_query(
            agent_id=agent_id,
            after_ts=after_ts,
            before_ts=before_ts,
            limit=k * 2,
            episode_type=episode_type,
        )
        for ep in episodes:
            self.sql.mark_accessed(ep.episode_id)

        # Score with mild query relevance via semantic
        hits = self.index.semantic_search(agent_id, query, k=len(episodes) or 1)
        relevance = {eid: sim for eid, sim in hits}

        return self._rerank(episodes, relevance)[:k]

    # ── Mode 3: Associative Retrieval ─────────────────────────────────

    def retrieve_associative(
        self,
        agent_id: str,
        query: str,
        seed_ids: Optional[List[str]] = None,
        k: Optional[int] = None,
        excluded_edge_types: Optional[Set[str]] = None,
    ) -> List[Tuple[Episode, float]]:
        """
        Graph spreading activation from seed episodes.
        If seed_ids not provided, uses top semantic hits as seeds.
        Implements Collins & Loftus (1975) spreading activation.
        """
        k = k or self.cfg.TOP_K_ASSOCIATIVE

        # Get seeds from semantic search if not provided
        if not seed_ids:
            hits = self.index.semantic_search(agent_id, query, k=3)
            seed_ids = [eid for eid, _ in hits]
            seed_acts = {eid: sim for eid, sim in hits}
        else:
            seed_acts = {sid: 1.0 for sid in seed_ids}

        if not seed_ids:
            return []

        # Ensure graph is loaded
        if not self.graph._loaded:
            self.graph.build_from_store(agent_id)

        # Spreading activation
        activated = self.graph.spreading_activation(
            seed_ids=seed_ids,
            seed_activations=seed_acts,
            excluded_types=excluded_edge_types,
        )

        if not activated:
            return []

        # Fetch activated episodes
        activated_ids = sorted(activated, key=lambda x: activated[x], reverse=True)
        episodes = self.sql.get_many(activated_ids[:k * 2])
        for ep in episodes:
            self.sql.mark_accessed(ep.episode_id)

        # Relevance = activation score (already incorporates graph distance)
        relevance = {ep.episode_id: activated.get(ep.episode_id, 0.0)
                     for ep in episodes}
        return self._rerank(episodes, relevance)[:k]

    # ── Mode 4: Cue-Based Retrieval ────────────────────────────────────

    def retrieve_cue_based(
        self,
        agent_id: str,
        query: str,
        who: Optional[List[str]] = None,
        where: Optional[str] = None,
        emotion: Optional[str] = None,
        emotion_valence_range: Optional[Tuple[float, float]] = None,
        episode_type: Optional[str] = None,
        tags: Optional[List[str]] = None,
        k: Optional[int] = None,
    ) -> List[Tuple[Episode, float]]:
        """
        Multi-facet cue-based retrieval using Tulving's encoding specificity.
        Any combination of: WHO, WHERE, EMOTION, TYPE, TAGS.
        """
        k = k or self.cfg.TOP_K_SEMANTIC

        val_min = emotion_valence_range[0] if emotion_valence_range else None
        val_max = emotion_valence_range[1] if emotion_valence_range else None

        episodes = self.sql.cue_query(
            agent_id=agent_id,
            who=who,
            where=where,
            emotion_label=emotion,
            emotion_valence_min=val_min,
            emotion_valence_max=val_max,
            episode_type=episode_type,
            tags=tags,
            limit=k * 2,
        )
        for ep in episodes:
            self.sql.mark_accessed(ep.episode_id)

        hits = self.index.semantic_search(agent_id, query, k=max(len(episodes), 1))
        relevance = {eid: sim for eid, sim in hits}
        return self._rerank(episodes, relevance)[:k]

    # ── Mode 5: Reconstructive Retrieval ──────────────────────────────

    def retrieve_reconstructive(
        self,
        agent_id: str,
        query: str,
        seed_episodes: Optional[List[Episode]] = None,
    ) -> Tuple[str, List[Tuple[Episode, float]]]:
        """
        Reconstructive recall: LLM rebuilds a coherent episode from fragments.
        Models Tulving (1983) — episodic memory is reconstructive, not verbatim playback.

        Returns (reconstructed_text, supporting_episodes).
        """
        Log.mode("reconstructive", f"[{agent_id}] '{query[:50]}'")

        # Gather fragments from multiple modes
        sem_results = self.retrieve_semantic(agent_id, query, k=3)
        assoc_results = self.retrieve_associative(agent_id, query, k=2)

        # Merge and deduplicate
        all_eps: Dict[str, Tuple[Episode, float]] = {}
        for ep, sc in sem_results + assoc_results:
            if ep.episode_id not in all_eps or sc > all_eps[ep.episode_id][1]:
                all_eps[ep.episode_id] = (ep, sc)

        fragments = list(all_eps.values())
        fragments.sort(key=lambda x: x[1], reverse=True)

        if not fragments:
            return "(no episodic fragments found)", []

        # Build fragment text for LLM
        frag_text = "\n\n".join(
            f"Fragment {i+1} [{ep.episode_type}, {ep.when_label}]:\n{ep.to_context_str()}"
            for i, (ep, _) in enumerate(fragments[:5])
        )

        # LLM reconstruction
        reconstructed = self._invoke(
            RECONSTRUCT_PROMPT, query=query, fragments=frag_text
        )

        Log.mode("reconstructive", f"reconstruction: {len(reconstructed)} chars")
        return reconstructed, fragments

    # ── Mode 6: Hybrid Retrieval ───────────────────────────────────────

    def retrieve_hybrid(
        self,
        agent_id: str,
        query: str,
        k: Optional[int] = None,
        use_reconstruction: bool = True,
    ) -> RetrievalResult:
        """
        Hybrid mode: run all modes, fuse results, detect interference.
        Final re-ranking uses composite_score over the merged pool.
        """
        k = k or self.cfg.TOP_K_SEMANTIC
        t0 = time.perf_counter()
        Log.section(f"HYBRID EPISODIC RETRIEVAL — '{query[:60]}'")

        # ── Run all four base modes ────────────────────────────────────
        sem_results   = self.retrieve_semantic(agent_id, query, k=k)
        temp_results  = self.retrieve_temporal(agent_id, query, days_back=14, k=k)
        assoc_results = self.retrieve_associative(agent_id, query, k=k)

        # ── Fuse results (union, keeping best score per episode) ───────
        fused: Dict[str, Tuple[Episode, float]] = {}
        for ep, sc in sem_results + temp_results + assoc_results:
            if ep.episode_id not in fused or sc > fused[ep.episode_id][1]:
                fused[ep.episode_id] = (ep, sc)

        merged = sorted(fused.values(), key=lambda x: x[1], reverse=True)
        top_k  = [ep for ep, _ in merged[:k]]
        scores = {ep.episode_id: sc for ep, sc in merged}

        # ── Interference detection ─────────────────────────────────────
        interference_pairs: List[Tuple[str, str]] = []
        for ep in top_k[:3]:
            conflicts = self.index.detect_interference(ep.episode_id, agent_id)
            for conf_id, sim in conflicts:
                pair = tuple(sorted([ep.episode_id, conf_id]))
                if pair not in [tuple(sorted(p)) for p in interference_pairs]:
                    interference_pairs.append((ep.episode_id, conf_id))
                    Log.mode("associative",
                             f"⚡ Interference: {ep.episode_id[:12]} ↔ {conf_id[:12]} "
                             f"(sim={sim:.3f})")

        # ── Reconstructive recall ──────────────────────────────────────
        reconstructed = None
        if use_reconstruction and top_k:
            reconstructed, _ = self.retrieve_reconstructive(
                agent_id, query, seed_episodes=top_k[:3]
            )

        # ── Narrative summary ──────────────────────────────────────────
        narrative = None
        if top_k:
            sorted_chrono = sorted(top_k, key=lambda e: e.when_ts)
            ep_text = "\n\n".join(ep.to_context_str() for ep in sorted_chrono[:5])
            narrative = self._invoke(NARRATIVE_PROMPT, episodes=ep_text)

        # ── KB retrieval ───────────────────────────────────────────────
        kb_docs = self.kb.retrieve(query, k=self.cfg.TOP_K_KB)

        elapsed_ms = int((time.perf_counter() - t0) * 1000)
        return RetrievalResult(
            query=query,
            agent_id=agent_id,
            mode=RetrievalMode.HYBRID,
            episodes=top_k,
            kb_docs=kb_docs,
            scores=scores,
            reconstructed=reconstructed,
            narrative=narrative,
            interference=interference_pairs,
            retrieval_time_ms=elapsed_ms,
            modes_used=["semantic", "temporal", "associative", "reconstructive"],
            total_searched=len(fused),
        )

    # ── Full Query Pipeline ────────────────────────────────────────────

    def query(
        self,
        agent_id: str,
        question: str,
        mode: RetrievalMode = RetrievalMode.HYBRID,
        **mode_kwargs,
    ) -> Dict[str, Any]:
        """
        Full query pipeline:
          1. Retrieve via chosen mode
          2. (Optional) Interference resolution
          3. Generate grounded answer with LLM
          4. Return structured result
        """
        t0 = time.time()
        Log.section(f"EPISODIC QUERY [{agent_id}] mode={mode.value} — '{question[:60]}'")

        # ── Retrieval dispatch ─────────────────────────────────────────
        result: RetrievalResult
        if mode == RetrievalMode.HYBRID:
            result = self.retrieve_hybrid(agent_id, question, **mode_kwargs)

        elif mode == RetrievalMode.SEMANTIC:
            eps_scores = self.retrieve_semantic(agent_id, question)
            eps   = [ep for ep, _ in eps_scores]
            scores = {ep.episode_id: sc for ep, sc in eps_scores}
            kb    = self.kb.retrieve(question)
            result = RetrievalResult(
                query=question, agent_id=agent_id, mode=mode,
                episodes=eps, kb_docs=kb, scores=scores,
                modes_used=["semantic"],
            )

        elif mode == RetrievalMode.TEMPORAL:
            eps_scores = self.retrieve_temporal(agent_id, question, **mode_kwargs)
            eps   = [ep for ep, _ in eps_scores]
            scores = {ep.episode_id: sc for ep, sc in eps_scores}
            kb    = self.kb.retrieve(question)
            result = RetrievalResult(
                query=question, agent_id=agent_id, mode=mode,
                episodes=eps, kb_docs=kb, scores=scores,
                modes_used=["temporal"],
            )

        elif mode == RetrievalMode.ASSOCIATIVE:
            eps_scores = self.retrieve_associative(agent_id, question, **mode_kwargs)
            eps   = [ep for ep, _ in eps_scores]
            scores = {ep.episode_id: sc for ep, sc in eps_scores}
            kb    = self.kb.retrieve(question)
            result = RetrievalResult(
                query=question, agent_id=agent_id, mode=mode,
                episodes=eps, kb_docs=kb, scores=scores,
                modes_used=["associative"],
            )

        elif mode == RetrievalMode.CUE_BASED:
            eps_scores = self.retrieve_cue_based(agent_id, question, **mode_kwargs)
            eps   = [ep for ep, _ in eps_scores]
            scores = {ep.episode_id: sc for ep, sc in eps_scores}
            kb    = self.kb.retrieve(question)
            result = RetrievalResult(
                query=question, agent_id=agent_id, mode=mode,
                episodes=eps, kb_docs=kb, scores=scores,
                modes_used=["cue_based"],
            )

        elif mode == RetrievalMode.RECONSTRUCTIVE:
            reconstructed, frag_scores = self.retrieve_reconstructive(
                agent_id, question
            )
            eps   = [ep for ep, _ in frag_scores]
            scores = {ep.episode_id: sc for ep, sc in frag_scores}
            kb    = self.kb.retrieve(question)
            result = RetrievalResult(
                query=question, agent_id=agent_id, mode=mode,
                episodes=eps, kb_docs=kb, scores=scores,
                reconstructed=reconstructed,
                modes_used=["semantic", "associative", "reconstructive"],
            )

        else:
            result = self.retrieve_hybrid(agent_id, question)

        # ── Log retrieved episodes ─────────────────────────────────────
        for ep in result.episodes[:5]:
            Log.episode(ep, score=result.scores.get(ep.episode_id, 0.0))

        # ── Interference resolution ────────────────────────────────────
        interference_resolved = []
        for ep_a_id, ep_b_id in result.interference[:2]:
            ep_a = self.sql.get_by_id(ep_a_id)
            ep_b = self.sql.get_by_id(ep_b_id)
            if ep_a and ep_b:
                raw = self._invoke(
                    INTERFERENCE_PROMPT,
                    id_a=ep_a_id, ep_a=ep_a.to_context_str(),
                    id_b=ep_b_id, ep_b=ep_b.to_context_str(),
                    query=question,
                )
                data = self._parse_json(raw, {})
                preferred = data.get("preferred_id", ep_a_id)
                reason    = data.get("reason", "")
                Log.warn(f"Interference resolved → {preferred[:12]}: {reason}")
                interference_resolved.append(preferred)

        # ── Format context for LLM ─────────────────────────────────────
        episodic_ctx = result.format_episodes_for_llm()
        kb_ctx = "\n\n".join(
            f"[{doc.metadata.get('source','?')}]\n{doc.page_content[:300]}"
            for doc in result.kb_docs
        ) or "(no KB results)"
        reconstructed = result.reconstructed or "(no reconstruction)"

        # ── Generate answer ────────────────────────────────────────────
        Log.step("Generating episodic-grounded answer")
        answer = self._invoke(
            ANSWER_PROMPT,
            query=question,
            episodic_context=episodic_ctx,
            kb_context=kb_ctx,
            reconstructed=reconstructed,
        )

        elapsed = round(time.time() - t0, 2)
        meta = {
            "mode":               mode.value,
            "episode_count":      len(result.episodes),
            "kb_count":           len(result.kb_docs),
            "graph_hops":         self.cfg.ACTIVATION_DEPTH if "associative" in result.modes_used else 0,
            "reconstructed":      result.reconstructed is not None,
            "interference_count": len(result.interference),
            "elapsed":            elapsed,
        }
        Log.answer_box(question, answer, meta)

        return {
            "question":     question,
            "answer":       answer,
            "agent_id":     agent_id,
            "mode":         mode.value,
            "result":       result,
            "interference": interference_resolved,
            "meta":         meta,
        }

    # ── Utilities ──────────────────────────────────────────────────────

    def show_timeline(self, agent_id: str, days_back: float = 14.0):
        """Print a chronological timeline of all episodes for an agent."""
        after = time.time() - days_back * 86400.0
        episodes = self.sql.temporal_query(
            agent_id=agent_id, after_ts=after, limit=50
        )
        # Reverse to show oldest first
        episodes = list(reversed(episodes))
        Log.section(f"EPISODE TIMELINE — [{agent_id}] last {days_back:.0f} days")
        for ep in episodes:
            age_d = (time.time() - ep.when_ts) / 86400.0
            print(
                f"  {C.DIM}{age_d:5.1f}d ago{C.RESET}  "
                f"{C.BOLD}{ep.episode_type:15s}{C.RESET}  "
                f"imp={ep.importance:.2f}  "
                f"emo={ep.emotion_label:12s}  "
                f"{ep.what[:55]}"
            )
        print(f"\n  {C.DIM}Total: {len(episodes)} episodes in window{C.RESET}")

    def show_graph_stats(self, agent_id: str):
        """Print episode graph statistics."""
        self.graph.build_from_store(agent_id)
        n_nodes = len(self.graph._adj)
        n_edges = sum(len(v) for v in self.graph._adj.values()) // 2
        Log.section(f"EPISODE GRAPH — [{agent_id}]")
        print(f"  Nodes (episodes): {n_nodes}")
        print(f"  Edges (links)   : {n_edges}")
        # Show edge type distribution
        type_counts: Dict[str, int] = {}
        for neighbors in self.graph._adj.values():
            for _, etype, _ in neighbors:
                type_counts[etype] = type_counts.get(etype, 0) + 1
        for etype, cnt in sorted(type_counts.items(), key=lambda x: -x[1]):
            print(f"  {C.DIM}  {etype:15s}: {cnt // 2} edges{C.RESET}")


In [23]:

# ══════════════════════════════════════════════════════════════════════
# DEMO QUERIES — one per retrieval mode + two hybrid
# ══════════════════════════════════════════════════════════════════════

DEMO_QUERIES = [
    # 1. Semantic — general similarity
    (
        "alex", RetrievalMode.SEMANTIC,
        "What technology decisions did Alex make recently?",
        "Mode: SEMANTIC — embedding similarity over all episodes",
        {},
    ),
    # 2. Temporal — last 7 days
    (
        "alex", RetrievalMode.TEMPORAL,
        "What has Alex been working on in the past week?",
        "Mode: TEMPORAL — time-window (days_back=7)",
        {"days_back": 7.0},
    ),
    # 3. Temporal — specific event type
    (
        "alex", RetrievalMode.TEMPORAL,
        "What decisions has Alex made recently?",
        "Mode: TEMPORAL — filtered to decision episodes only",
        {"days_back": 14.0, "episode_type": "decision"},
    ),
    # 4. Associative — spreading activation from Kafka frustration
    (
        "alex", RetrievalMode.ASSOCIATIVE,
        "What followed the Kafka production incident?",
        "Mode: ASSOCIATIVE — spreading activation from emotion_event → resolution",
        {},
    ),
    # 5. Cue-based — negative emotion episodes
    (
        "alex", RetrievalMode.CUE_BASED,
        "When was Alex frustrated or upset?",
        "Mode: CUE_BASED — emotion_label=negative filter",
        {"emotion": "negative"},
    ),
    # 6. Cue-based — tag filter
    (
        "alex", RetrievalMode.CUE_BASED,
        "What has Alex done related to RAG and vector databases?",
        "Mode: CUE_BASED — tags=['rag', 'vector_db']",
        {"tags": ["rag"]},
    ),
    # 7. Reconstructive — rebuild Kafka incident from fragments
    (
        "alex", RetrievalMode.RECONSTRUCTIVE,
        "Reconstruct what happened with the Kafka consumer lag issue — "
        "what caused it, how Alex felt, and how it was resolved.",
        "Mode: RECONSTRUCTIVE — rebuild full episode from fragments",
        {},
    ),
    # 8. Hybrid — full pipeline
    (
        "alex", RetrievalMode.HYBRID,
        "What should Alex focus on next, given his recent work on RAG deployment "
        "and upcoming system design interview?",
        "Mode: HYBRID — all modes fused, interference detection, narrative, KB grounding",
        {"use_reconstruction": True},
    ),
]


In [24]:
w =76

In [25]:

# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config, agent_id: str = "alex") -> EpisodicRetrievalEngine:
    engine = EpisodicRetrievalEngine(cfg)
    engine.setup()

    # Load demo episodes if none exist for this agent
    count = engine.sql.count_for_agent(agent_id)
    if count == 0:
        Log.step(f"No existing episodes for [{agent_id}] — loading demo history")
        engine.load_demo_episodes(agent_id)
    else:
        Log.ok(f"[{agent_id}] {count} existing episodes loaded from store")
        # Rebuild graph from persisted edges
        engine.graph.build_from_store(agent_id)

    # Print reference table
    Log.step("Reference papers / datasets:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    return engine



In [26]:

# ══════════════════════════════════════════════════════════════════════
# RUN MODES
# ══════════════════════════════════════════════════════════════════════

def run_demo(engine: EpisodicRetrievalEngine):
    """Run all 8 demo queries, one per retrieval mode."""
    print(f"\n{C.BOLD}{C.CYAN}Running {len(DEMO_QUERIES)} demo queries "
          f"(all 6 retrieval modes)…{C.RESET}")

    for i, (agent_id, mode, query, note, kwargs) in enumerate(DEMO_QUERIES, 1):
        print(f"\n{C.BOLD}{C.YELLOW}{'▓'*W}{C.RESET}")
        print(f"{C.BOLD}{C.YELLOW}  Demo {i}/{len(DEMO_QUERIES)}: {note}{C.RESET}")
        print(f"{C.BOLD}{C.YELLOW}{'▓'*W}{C.RESET}")
        try:
            engine.query(agent_id, query, mode=mode, **kwargs)
        except Exception as exc:
            Log.err(f"Demo {i} failed: {exc}")
        time.sleep(0.3)


def run_single_demo(engine: EpisodicRetrievalEngine, n: int):
    if not (1 <= n <= len(DEMO_QUERIES)):
        Log.err(f"--demo-n must be 1–{len(DEMO_QUERIES)}")
        sys.exit(1)
    agent_id, mode, query, note, kwargs = DEMO_QUERIES[n - 1]
    print(f"\n{C.DIM}Demo {n}: {note}{C.RESET}")
    engine.query(agent_id, query, mode=mode, **kwargs)


def run_interactive(engine: EpisodicRetrievalEngine, agent_id: str = "alex"):
    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════════╗")
    print("║      EPISODIC MEMORY RETRIEVAL RAG — Interactive CLI             ║")
    print("║  Commands:                                                       ║")
    print("║    'mode <name>'  — set retrieval mode (semantic/temporal/       ║")
    print("║                    associative/cue_based/reconstructive/hybrid)  ║")
    print("║    'timeline'     — show episode timeline                        ║")
    print("║    'graph'        — show graph statistics                        ║")
    print("║    'store <text>' — store a new episode from free text           ║")
    print("║    'agent <id>'   — switch agent                                 ║")
    print("║    'refs'         — print paper references                       ║")
    print("║    'demo N'       — run demo query N                             ║")
    print("║    'quit' / 'q'   — exit                                         ║")
    print("╚══════════════════════════════════════════════════════════════════╝")
    print(C.RESET)

    current_mode = RetrievalMode.HYBRID
    current_agent = agent_id

    while True:
        try:
            user = input(
                f"\n{C.BOLD}[{current_agent}][{current_mode.value}] Query:{C.RESET} "
            ).strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue
        cmd = user.lower()

        if cmd in ("quit", "exit", "q"):
            break

        if cmd == "timeline":
            engine.show_timeline(current_agent)
            continue

        if cmd == "graph":
            engine.show_graph_stats(current_agent)
            continue

        if cmd == "refs":
            for i, (t, u) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i}] {t} — {u}")
            continue

        if cmd.startswith("mode "):
            mode_str = user[5:].strip().lower()
            try:
                current_mode = RetrievalMode(mode_str)
                print(f"  Mode set to: {current_mode.value}")
            except ValueError:
                Log.warn(f"Unknown mode '{mode_str}'. "
                         f"Valid: {[m.value for m in RetrievalMode]}")
            continue

        if cmd.startswith("agent "):
            current_agent = user[6:].strip()
            count = engine.sql.count_for_agent(current_agent)
            if count == 0:
                Log.warn(f"No episodes for [{current_agent}]. "
                         f"Use 'store <text>' to add some.")
            else:
                engine.graph.build_from_store(current_agent)
                Log.ok(f"Switched to [{current_agent}] — {count} episodes")
            continue

        if cmd.startswith("store "):
            text = user[6:].strip()
            if text:
                engine.store_episode(current_agent, text)
            continue

        if cmd.startswith("demo"):
            parts = cmd.split()
            if len(parts) > 1 and parts[1].isdigit():
                run_single_demo(engine, int(parts[1]))
            else:
                print(f"  Demo queries (1–{len(DEMO_QUERIES)}):")
                for i, (_, mode, q, note, _) in enumerate(DEMO_QUERIES, 1):
                    print(f"  {i}. [{mode.value}] {q[:60]}…")
            continue

        # Parse inline cue flags for cue_based mode
        kwargs = {}
        if current_mode == RetrievalMode.CUE_BASED:
            # Simple parsing: --tags rag,chromadb --emotion positive --days 7
            import re
            tags_m = re.search(r'--tags\s+([\w,]+)', user)
            emot_m = re.search(r'--emotion\s+(\w+)', user)
            days_m = re.search(r'--days\s+([\d.]+)', user)
            if tags_m:
                kwargs["tags"] = tags_m.group(1).split(",")
                user = re.sub(r'--tags\s+[\w,]+', '', user).strip()
            if emot_m:
                kwargs["emotion"] = emot_m.group(1)
                user = re.sub(r'--emotion\s+\w+', '', user).strip()
            if days_m:
                kwargs["days_back"] = float(days_m.group(1))
                user = re.sub(r'--days\s+[\d.]+', '', user).strip()
        elif current_mode == RetrievalMode.TEMPORAL:
            import re
            days_m = re.search(r'--days\s+([\d.]+)', user)
            if days_m:
                kwargs["days_back"] = float(days_m.group(1))
                user = re.sub(r'--days\s+[\d.]+', '', user).strip()

        try:
            engine.query(current_agent, user, mode=current_mode, **kwargs)
        except Exception as exc:
            Log.err(f"Error: {exc}")
